In [1]:
!pip install -q ultralytics deep-sort-realtime flask flask-sock flask-cors pyngrok easyocr rapidfuzz scikit-image faiss-cpu kaggle


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 91.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is

In [2]:
import os, base64, time, threading, queue, json, socket as _socket, pickle, re, glob
import numpy as np
import cv2
from PIL import Image
from flask import Flask, Response, jsonify, request
from flask_sock import Sock
from flask_cors import CORS
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from datetime import datetime
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
import torch, timm
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import faiss
from skimage.feature import local_binary_pattern
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
import warnings

warnings.filterwarnings('ignore')

try:
    import easyocr
    from rapidfuzz import process as rfuzz_process, fuzz
    OCR_IMPORT_OK = True
except ImportError:
    OCR_IMPORT_OK = False

log = lambda m: print(f"[{datetime.now().strftime('%H:%M:%S')}] {m}")
try:
    torch.zeros(1).cuda()
    device = 'cuda'
except Exception:
    device = 'cpu'
    print("CUDA unavailable, using CPU")
log(f"Device: {device}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[11:13:19] Device: cuda


In [3]:
# ══ Auto-download latest gallery from Kaggle ═════════════════════════════════
import kaggle as _kaggle_mod

_GALLERY_DATASET      = "abdelghaniyacine/gallary-dino-76"
_GALLERY_DOWNLOAD_DIR = "/kaggle/working/gallary-dino-76"
_GALLERY_FALLBACK_DIR = "/kaggle/input/datasets/abdelghaniyacine/gallary-dino-76"

def _gallery_path(filename):
    working  = f"{_GALLERY_DOWNLOAD_DIR}/{filename}"
    fallback = f"{_GALLERY_FALLBACK_DIR}/{filename}"
    return working if os.path.exists(working) else fallback

def download_latest_gallery():
    log(f"Downloading latest gallery: {_GALLERY_DATASET} ...")
    try:
        _kaggle_mod.api.authenticate()
        _kaggle_mod.api.dataset_download_files(
            _GALLERY_DATASET,
            path=_GALLERY_DOWNLOAD_DIR,
            unzip=True,
            force=True,
            quiet=False
        )
        log(f"  ✅ Gallery downloaded → {_GALLERY_DOWNLOAD_DIR}")
        return True
    except Exception as e:
        log(f"  ⚠️ Download failed ({e}) — using /kaggle/input/ fallback")
        return False

download_latest_gallery()

[11:13:20] Downloading latest gallery: abdelghaniyacine/gallary-dino-76 ...
Dataset URL: https://www.kaggle.com/datasets/abdelghaniyacine/gallary-dino-76


100%|██████████| 1.10G/1.10G [00:06<00:00, 184MB/s] 



[11:13:35]   ✅ Gallery downloaded → /kaggle/working/gallary-dino-76


True

In [4]:
# ══ Tracking config  (yolo-deepsort-cnn-v1-version-stable) ══════════════════
CONF_THRESHOLD           = 0.25
MAX_LOST_AGE             = 90
REID_MIN_SCORE           = 0.72
JPEG_QUALITY             = 65
TARGET_FPS               = 20
EXITED_RAW_TTL           = 300
EXIT_MARGIN_STATIC       = 0.05
EXIT_MARGIN_DYNAMIC      = 0.12
MIN_FRAMES_BEFORE_LOST   = 15
MAX_REALISTIC_SPEED      = 15.0

# ══ Recognition-adapter tuning  (yolo-deepsort-cnn-v1-version-stable) ═══════
RECOGNITION_CONF_LOCK        = 0.72
RECOGNITION_REQUERY_INTERVAL = 15
RECOGNITION_MAX_VOTES        = 7

# ══ Paths ════════════════════════════════════════════════════════════════════
_yolo_exact = "/kaggle/input/datasets/sarahlaouedj25/yoloresultobbv5/best(2).pt"
_yolo_glob  = glob.glob("/kaggle/input/**/best*.pt", recursive=True)
YOLO_PT     = _yolo_exact if os.path.exists(_yolo_exact) else (_yolo_glob[0] if _yolo_glob else _yolo_exact)

FINETUNED_CHECKPOINT = _gallery_path("dinov2_finetuned_supcon_v3.pt")
GALLERY_PKL          = _gallery_path("gallery_finetuned_v3.pkl")
FAISS_PATH_GALLERY   = _gallery_path("gallery_finetuned_v3.index")
ENSEMBLE_PKL         = "/kaggle/working/ensemble_v36.pkl"
RELOAD_SECRET        = os.environ.get("RELOAD_SECRET", "smartbasket-reload-secret")

# ══ dino-v36 hyperparameters (verbatim) ══════════════════════════════════════
YOLO_CONF_THR        = 0.25
IMAGE_EXTENSIONS     = {'.jpg','.jpeg','.png','.bmp','.webp','.tiff'}
K_NEIGHBORS          = 30
TOP_K_SHOW           = 3
OBB_CROP_SIZE        = 336
TTA_ANGLES           = [0, 45, 90, 135, 180, 270]
TTA_FLIP             = True
MARGIN_WEIGHT        = 2.0
SINGLETON_THR_FACTOR = 0.88
THR_HARD_CAP         = 0.82
THR_ABSOLUTE_MAX     = 0.92
GLOBAL_THR_FACTOR    = 0.95
MARGIN_MIN_FALLBACK  = 0.03
FALLBACK_BASE_KNN    = 0.70

ENS_N_TOP            = 5
ENS_MIN_PROB         = 0.48
RF_N_ESTIMATORS      = 400
ET_N_ESTIMATORS      = 400
RF_MIN_SAMPLES_LEAF  = 2
ET_MIN_SAMPLES_LEAF  = 2
ENS_HARD_NEG_WEIGHT  = 3.0
STACK_C              = 2.0
STACK_MAX_ITER       = 1000

OCR_LANGS            = ['ar', 'en']
OCR_MIN_PARTIAL      = 55
OCR_MIN_TOKEN_SORT   = 60
OCR_HIGH_CONF        = 80
OCR_MID_CONF         = 55
OCR_VETO_MIN_CONF    = 0.5
OCR_VETO_MAX_PROB    = 0.80
OCR_EXACT_TOKEN_THR  = 90
OCR_BOOST_THR        = 70

HSV_HR_BINS_H        = 64
HSV_HR_BINS_S        = 32
HSV_HR_BINS_V        = 16

UNKNOWN_BOOST_DEFAULT  = 1.12
UNKNOWN_BOOST_MAX      = 1.45
UNKNOWN_STRICT_MARGIN  = 0.000
UNKNOWN_STRICT_KNN     = 0.75
UNKNOWN_STRICT_BOOST   = 1.25
UNKNOWN_STRICT2_KNN    = 0.820
UNKNOWN_STRICT2_MARGIN = 0.012

HG_CLUSTER_THRESHOLD = 0.90
HG_MIN_CLUSTER_SIZE  = 2

MARGIN_THR_SCALE     = 0.40
MARGIN_THR_REF       = 0.15

OOD_GAP_REJECT_THR   = 0.04
OOD_MARGIN_REJECT    = 0.05
OOD_KNN_REJECT       = 0.82
OOD_TARGET_FPR       = 0.03
OOD_MIN_FLOOR        = 0.01

HSV_BONUS_MAX  = 0.08
HSV_PENALTY    = 0.04
PAD_SMALL=0.12; PAD_MID=0.07; PAD_LARGE=0.03
PAD_SMALL_THR=0.08; PAD_LARGE_THR=0.25

FEAT_NAMES = [
    'knn','cen','margin','rank','n_img','hsv','lbp','ratio',
    'ocr_conf','ocr_comb','same_hard_group','box_wh_ratio','box_area_frac',
    'knn_gap_23','rank_norm','cen_rank','intra_grp_rank','ocr_exact',
    'std_top5','centroid_gap','margin_ratio','rank_consistency',
]
N_FEATURES = len(FEAT_NAMES)

In [5]:
# ══ DINOv2 model loading (dino-v36 logic verbatim) ═══════════════════════════
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
tfm224 = T.Compose([T.Resize((224,224),interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(224),T.ToTensor(),T.Normalize(MEAN,STD)])
tfm336 = T.Compose([T.Resize((336,336),interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(336),T.ToTensor(),T.Normalize(MEAN,STD)])

log("Loading DINOv2...")

def _timm_ns(name):
    orig = torch.nn.Module.load_state_dict
    def _p(self, sd, strict=True, assign=False): return orig(self, sd, strict=False, assign=assign)
    torch.nn.Module.load_state_dict = _p
    try: return timm.create_model(name, pretrained=True, num_classes=0, global_pool='avg')
    finally: torch.nn.Module.load_state_dict = orig

dinov2 = None
for fn in [
    lambda: torch.hub.load('facebookresearch/dinov2','dinov2_vitl14_reg',pretrained=True,force_reload=False),
    lambda: _timm_ns('vit_large_patch14_reg4_dinov2.lvd142m'),
]:
    try: dinov2 = fn(); log("  DINOv2 base OK"); break
    except Exception as e: log(f"  {e}")
if dinov2 is None: raise RuntimeError("DINOv2 failed to load")
dinov2 = dinov2.to(device).eval()
for p in dinov2.parameters(): p.requires_grad = False

if os.path.exists(FINETUNED_CHECKPOINT):
    log(f"Loading finetuned checkpoint...")
    try:
        ckpt = torch.load(FINETUNED_CHECKPOINT, map_location=device)
        missing, _ = dinov2.load_state_dict(ckpt['backbone_sd'], strict=False)
        if missing: log(f"  WARN: {len(missing)} missing keys")
        dinov2.eval()
        for p in dinov2.parameters(): p.requires_grad = False
        log("  Finetuned backbone loaded ✓")
    except Exception as e:
        log(f"  ERROR: {e}")
else:
    log("  Finetuned checkpoint not found — using base DINOv2")

@torch.no_grad()
def enc(pil_img):
    img = pil_img.convert('RGB')
    t224 = tfm224(img).unsqueeze(0).to(device)
    f224 = F.normalize(dinov2(t224), p=2, dim=1)
    if use_multiscale:
        t336 = tfm336(img).unsqueeze(0).to(device)
        f336 = F.normalize(dinov2(t336), p=2, dim=1)
        return F.normalize(torch.cat([f224,f336],dim=1),p=2,dim=1).cpu().numpy()[0].astype(np.float32)
    return f224.cpu().numpy()[0].astype(np.float32)

def encode_tta(crop):
    if crop is None: return None
    embs = []
    for a in TTA_ANGLES:
        for fl in ([False,True] if TTA_FLIP else [False]):
            img = crop.copy()
            if a: img = img.rotate(a, expand=False, resample=Image.BICUBIC, fillcolor=(128,128,128))
            if fl: img = img.transpose(Image.FLIP_LEFT_RIGHT)
            embs.append(enc(img))
    avg = np.mean(embs, axis=0).astype(np.float32)
    n = np.linalg.norm(avg)
    return avg/n if n > 0 else avg

[11:13:41] Loading DINOv2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_reg4_pretrain.pth


100%|██████████| 1.13G/1.13G [01:04<00:00, 19.0MB/s]


[11:14:52]   DINOv2 base OK
[11:14:52] Loading finetuned checkpoint...
[11:14:53]   Finetuned backbone loaded ✓


In [6]:
# ══ Visual / OCR / math helper functions (dino-v36 verbatim) ════════════════

def compute_centroid_gap(emb, cen_sims_sorted_vals):
    if len(cen_sims_sorted_vals) < 2: return 1.0
    sim1 = float(cen_sims_sorted_vals[0])
    sim2 = float(cen_sims_sorted_vals[1])
    return float(np.clip((sim1-sim2)/(sim1+sim2+1e-8), 0.0, 1.0))

def compute_margin_ratio(margin, lbl, median_margins):
    med = median_margins.get(lbl, 0.05)
    if med < 1e-6: return 1.0
    return float(np.clip(margin / med, 0.0, 5.0))

def compute_rank_consistency(pred_lbl, knn_rank, cen_rank_map):
    cen_rk = cen_rank_map.get(pred_lbl, 50)
    discrepancy = abs(knn_rank - min(cen_rk, 10))
    return float(np.clip(1.0 - discrepancy / 10.0, 0.0, 1.0))

def extract_features_v36(knn_sim, centroid_sim, margin, rank, n_images,
                          hsv_sim, lbp_sim, knn_sim_top1,
                          ocr_conf, ocr_match_flag, same_hard_group,
                          box_wh_ratio=1.0, box_area_frac=0.1,
                          knn_gap_23=0.0, rank_norm=0.0, cen_rank=0,
                          intra_grp_rank=99, ocr_exact=0, std_top5=0.05,
                          centroid_gap=0.5, margin_ratio=1.0, rank_consistency=1.0):
    ratio = knn_sim / (knn_sim_top1 + 1e-6)
    return [
        float(knn_sim), float(centroid_sim), float(margin),
        float(rank), float(n_images), float(hsv_sim), float(lbp_sim), float(ratio),
        float(ocr_conf), float(ocr_conf*ocr_match_flag),
        float(same_hard_group), float(box_wh_ratio), float(box_area_frac),
        float(knn_gap_23), float(rank_norm), float(cen_rank),
        float(min(intra_grp_rank,10)), float(ocr_exact), float(std_top5),
        float(centroid_gap), float(margin_ratio), float(rank_consistency),
    ]

def extract_color_hist(pil_img, bins_h=24, bins_s=12, bins_v=0):
    rgb = np.array(pil_img.convert('RGB').resize((128,128)))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h = cv2.calcHist([hsv],[0],None,[bins_h],[0,180]).flatten()
    s = cv2.calcHist([hsv],[1],None,[bins_s],[0,256]).flatten()
    parts = [h, s]
    if bins_v > 0:
        v = cv2.calcHist([hsv],[2],None,[bins_v],[0,256]).flatten()
        parts.append(v)
    hist = np.concatenate(parts).astype(np.float32)
    n = hist.sum()
    return hist/n if n > 0 else hist

def chi2_distance(h1, h2):
    denom = h1 + h2 + 1e-10
    return float(np.sum((h1-h2)**2/denom))

def extract_lbp(pil_img, radius=2, n_points=16, size=64):
    gray = np.array(pil_img.convert('L').resize((size,size)))
    lbp = local_binary_pattern(gray, n_points, radius, method='uniform')
    hist,_ = np.histogram(lbp.ravel(), bins=n_points+2, range=(0,n_points+2))
    hist = hist.astype(np.float32); n = hist.sum()
    return hist/n if n > 0 else hist

def hsv_similarity_hr(pil_img, lbl):
    if pil_img is None or lbl not in visual_sigs: return 0.5
    sig = visual_sigs[lbl]
    hist_hr = extract_color_hist(pil_img, HSV_HR_BINS_H, HSV_HR_BINS_S, HSV_HR_BINS_V)
    sig_hr = sig.get('color_hr', sig['color'])
    if len(hist_hr) != len(sig_hr):
        sig_hr = sig['color']; hist_hr = extract_color_hist(pil_img)
    d = chi2_distance(hist_hr, sig_hr)
    return float(1.0 / (1.0 + d * 5.0))

def get_visual_scores(crop_pil, lbl):
    if crop_pil is None or lbl not in visual_sigs: return 0.5, 0.5
    sig = visual_sigs[lbl]
    hsv_s = float(np.dot(extract_color_hist(crop_pil), sig['color']))
    lbp_s = float(np.dot(extract_lbp(crop_pil), sig['lbp']))
    return hsv_s, lbp_s

def preprocess_for_ocr(pil_img):
    img = np.array(pil_img.convert('RGB'))
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

def clean_ocr_text(raw_texts):
    cleaned = []
    for text in raw_texts:
        t = re.sub(r'[\xc2\xa7\xa9\xae\u2122?!|\[\]{}\xb0\u2022\u2192\u2190]','',text)
        t = re.sub(r'[0-9]{7,}','',t)
        t = re.sub(r'\s+',' ',t).strip()
        if len(t)>=3 and not t.replace('.','').replace(',','').isdigit():
            cleaned.append(t)
    return ' '.join(cleaned)

def _get_family_filter(text_norm, top5_labels):
    words = re.findall(r'[a-z]{3,}', text_norm)
    if not words: return top5_labels
    keyword_matches = []
    for prod in gallery_prods:
        prod_norm = re.sub(r'[^a-z0-9]','',prod.lower())
        for w in words:
            if len(w)>=4 and w in prod_norm:
                keyword_matches.append(prod); break
    if 2<=len(keyword_matches)<=max(2,len(gallery_prods)//3):
        return list(set(top5_labels)|set(keyword_matches[:15]))
    return top5_labels

def ocr_match_full(crop_pil, top5_labels):
    if not OCR_AVAILABLE or crop_pil is None:
        return None, 0.0, 'none', 0, ''
    try:
        raw = ocr_reader.readtext(np.array(preprocess_for_ocr(crop_pil)), detail=0, paragraph=True)
        text = clean_ocr_text(raw)
    except Exception:
        return None, 0.0, 'none', 0, ''
    if not text or len(text)<3: return None, 0.0, 'none', 0, ''
    text_norm = re.sub(r'[^a-z0-9]','',text.lower())
    if len(text_norm)<3: return None, 0.0, 'none', 0, ''
    search_pool = _get_family_filter(text_norm, top5_labels)
    candidates  = {l: GALLERY_NAMES_NORM[l] for l in search_pool if l in GALLERY_NAMES_NORM}
    if not candidates:
        candidates = {l: GALLERY_NAMES_NORM[l] for l in gallery_prods if l in GALLERY_NAMES_NORM}
    r1 = rfuzz_process.extractOne(text_norm, candidates, scorer=fuzz.partial_ratio,    score_cutoff=OCR_MIN_PARTIAL)
    r2 = rfuzz_process.extractOne(text_norm, candidates, scorer=fuzz.token_sort_ratio, score_cutoff=OCR_MIN_TOKEN_SORT)
    best_lbl, best_score, method, ts_score = None, 0, 'none', 0
    if r1 and r1[1]>=best_score: best_lbl, best_score, method = r1[0], r1[1], 'partial'
    if r2 and r2[1]>=best_score:
        best_lbl, best_score, method, ts_score = r2[0], r2[1], 'token_sort', int(r2[1])
    if best_lbl is None:
        all_cands = {l: GALLERY_NAMES_NORM[l] for l in gallery_prods if l in GALLERY_NAMES_NORM}
        r3 = rfuzz_process.extractOne(text_norm, all_cands, scorer=fuzz.token_sort_ratio, score_cutoff=OCR_MIN_TOKEN_SORT)
        if r3: best_lbl, best_score, method, ts_score = r3[0], r3[1], 'token_sort_global', int(r3[1])
    if best_lbl is None: return None, 0.0, 'none', 0, text_norm
    if best_score>=OCR_HIGH_CONF: return best_lbl, 1.0, method, ts_score, text_norm
    elif best_score>=OCR_MID_CONF: return best_lbl, 0.5, method, ts_score, text_norm
    return None, 0.0, 'none', 0, text_norm

def compute_hard_groups(centroids, threshold=HG_CLUSTER_THRESHOLD, min_size=HG_MIN_CLUSTER_SIZE):
    labels_list = sorted(centroids.keys())
    if len(labels_list) < 2: return {}, {}
    C = np.stack([centroids[l] for l in labels_list]).astype(np.float32)
    sim_matrix  = C @ C.T
    np.fill_diagonal(sim_matrix, 1.0)
    dist_matrix = np.clip(1.0 - sim_matrix, 0.0, 2.0)
    clust = AgglomerativeClustering(
        n_clusters=None, distance_threshold=1.0 - threshold,
        metric='precomputed', linkage='average')
    clust.fit(dist_matrix)
    cluster_to_labels = defaultdict(list)
    for lbl, cid in zip(labels_list, clust.labels_):
        cluster_to_labels[int(cid)].append(lbl)
    hard_groups = {}; label_to_hard_groups = defaultdict(list); gid = 0
    for members in cluster_to_labels.values():
        if len(members) < min_size: continue
        gname = f"hg_{gid}"
        hard_groups[gname] = members
        for m in members: label_to_hard_groups[m].append(gname)
        gid += 1
    log(f"  {len(hard_groups)} hard groups")
    return dict(hard_groups), dict(label_to_hard_groups)

In [7]:
# ══ Gallery + FAISS loading (dino-v36 logic) ═════════════════════════════════
log("Loading gallery PKL...")
with open(GALLERY_PKL,'rb') as f: gdata = pickle.load(f)

gallery_labels      = gdata.get('all_labels', gdata['labels'])
gallery_labels_orig = gdata.get('labels', gallery_labels)
DIM                 = gdata['embed_dim']
use_multiscale      = gdata.get('use_multiscale', False)
centroids           = gdata.get('centroids', {})
visual_sigs         = gdata.get('visual_signatures', {})
global_thr_raw      = gdata.get('global_thr', 0.734)
per_label_thr_raw   = gdata.get('per_label_thr', {})
label_counts        = gdata.get('label_counts', {})
label_to_idxs       = gdata.get('label_to_idxs', {})
gallery_prods       = gdata.get('products', sorted(set(gallery_labels)))

log(f"  {len(gallery_prods)} products, {len(gallery_labels)} TTA vectors, dim={DIM}")

log("Loading FAISS index...")
faiss_index = faiss.read_index(FAISS_PATH_GALLERY)
try:
    faiss_index.make_direct_map()
except Exception:
    pass
log(f"  FAISS: {faiss_index.ntotal} vectors")

# ── Per-label threshold processing ───────────────────────────────────────────
per_label_thr = {}
for lbl, thr in per_label_thr_raw.items():
    cnt = label_counts.get(lbl, 1)
    t = thr * SINGLETON_THR_FACTOR if cnt == 1 else thr
    t = min(t, THR_HARD_CAP)
    per_label_thr[lbl] = round(t, 4)
global_thr = round(min(global_thr_raw * GLOBAL_THR_FACTOR, THR_HARD_CAP), 4)

# ── Centroid matrix ───────────────────────────────────────────────────────────
cen_labels_global  = sorted(centroids.keys())
cen_matrix_global  = np.stack([centroids[l] for l in cen_labels_global]).astype(np.float32)
cen_matrix         = cen_matrix_global
cen_labels         = cen_labels_global

# ── Hard groups ───────────────────────────────────────────────────────────────
hard_groups, label_to_hard_groups = compute_hard_groups(centroids)

group_coMembers = defaultdict(set)
for gname, members in hard_groups.items():
    for m in members:
        for other in members:
            if other != m: group_coMembers[m].add(other)

GALLERY_NAMES_NORM = {lbl: re.sub(r'[^a-z0-9]','',lbl.lower()) for lbl in gallery_prods}

# ── Reconstruct embedding matrix from FAISS ───────────────────────────────────
log("Reconstructing embedding matrix from FAISS...")
_n = faiss_index.ntotal
emb_mat = np.zeros((_n, DIM), dtype=np.float32)
for i in range(_n):
    try: emb_mat[i] = faiss_index.reconstruct(i)
    except Exception: pass
log(f"  emb_mat: {emb_mat.shape}")

# ── Median margins ────────────────────────────────────────────────────────────
log("Computing median margins...")
median_margins = {}
for lbl, idxs in label_to_idxs.items():
    if len(idxs) < 2: median_margins[lbl] = 0.05; continue
    margins = []
    for idx in idxs:
        emb = emb_mat[idx].astype(np.float32)
        sims_all = cen_matrix @ emb
        sorted_sims = np.sort(sims_all)[::-1]
        margin = float(sorted_sims[0] - sorted_sims[1]) if len(sorted_sims) > 1 else 0.05
        margins.append(margin)
    median_margins[lbl] = float(np.median(margins))

# ── Gap scores for OOD calibration ───────────────────────────────────────────
gap_scores_known = []
for lbl, idxs in label_to_idxs.items():
    if lbl not in centroids: continue
    for idx in idxs:
        emb = emb_mat[idx].astype(np.float32)
        sims_all = cen_matrix @ emb
        sorted_sims = np.sort(sims_all)[::-1]
        gap = compute_centroid_gap(emb, sorted_sims)
        gap_scores_known.append(gap)
gap_arr = np.array(gap_scores_known)
log(f"  centroid_gap known: mean={gap_arr.mean():.4f} p5={np.percentile(gap_arr,5):.4f}")

[11:14:53] Loading gallery PKL...
[11:14:53]   313 products, 11199 TTA vectors, dim=1024
[11:14:53] Loading FAISS index...
[11:14:53]   FAISS: 11199 vectors
[11:14:53]   52 hard groups
[11:14:53] Reconstructing embedding matrix from FAISS...
[11:14:53]   emb_mat: (11199, 1024)
[11:14:53] Computing median margins...
[11:14:53]   centroid_gap known: mean=0.0665 p5=0.0081


In [8]:
# ══ Calibration functions + build_ensemble_v36 (dino-v36 verbatim) ══════════

def calibrate_ood_gap_threshold(emb_mat, gallery_labels_orig, label_to_idxs,
                                  centroids, cen_labels, cen_matrix,
                                  gap_scores_known, target_fpr=OOD_TARGET_FPR):
    log(f"Calibrating centroid_gap threshold (target FPR={target_fpr})...")
    rng = np.random.default_rng(42)
    labels_list = list(label_to_idxs.keys())
    gap_scores_unknown = []
    n_fake = len(gap_scores_known)
    for _ in range(n_fake):
        l1, l2 = rng.choice(labels_list, 2, replace=False)
        idxs1 = label_to_idxs.get(l1, []); idxs2 = label_to_idxs.get(l2, [])
        if not idxs1 or not idxs2: continue
        i1 = int(rng.choice(idxs1)); i2 = int(rng.choice(idxs2))
        alpha = rng.uniform(0.35, 0.65)
        fake = alpha * emb_mat[i1] + (1.0 - alpha) * emb_mat[i2]
        n = np.linalg.norm(fake)
        if n < 1e-8: continue
        fake = (fake / n).astype(np.float32)
        sims_all = cen_matrix @ fake
        sorted_sims = np.sort(sims_all)[::-1]
        gap = compute_centroid_gap(fake, sorted_sims)
        gap_scores_unknown.append(gap)
    if not gap_scores_unknown: return 0.02
    gap_known = np.array(gap_scores_known, dtype=np.float32)
    gap_unknown = np.array(gap_scores_unknown, dtype=np.float32)
    log(f"  gap_known  : mean={gap_known.mean():.4f} p5={np.percentile(gap_known,5):.4f}")
    log(f"  gap_unknown: mean={gap_unknown.mean():.4f} p95={np.percentile(gap_unknown,95):.4f}")
    best_thr = OOD_MIN_FLOOR; best_score = -1.0
    for pct in range(1, 20):
        thr = float(np.percentile(gap_known, pct))
        if thr < OOD_MIN_FLOOR: continue
        fpr = float(np.mean(gap_known < thr))
        tpr_unk = float(np.mean(gap_unknown < thr))
        if fpr <= target_fpr:
            score = tpr_unk - fpr * 2
            if score > best_score: best_score = score; best_thr = thr
    log(f"  OOD gap threshold = {best_thr:.4f}")
    return float(best_thr)


def calibrate_unknown_boost(rf_model, et_model, stack_model, scaler,
                              embeddings_orig, gallery_labels_orig,
                              label_to_idxs_orig, centroids, label_counts,
                              label_to_hard_groups, median_margins,
                              cen_labels, cen_matrix):
    log("Calibrating unknown boost...")
    fp_rate = {}
    col_pos_rf_c = list(rf_model.classes_).index(1) if 1 in rf_model.classes_ else 1
    col_pos_et_c = list(et_model.classes_).index(1) if 1 in et_model.classes_ else 1
    for lbl, idxs in label_to_idxs_orig.items():
        if len(idxs)<2: continue
        false_pos=0; total_neg=0
        for query_idx in idxs:
            query_emb  = embeddings_orig[query_idx]
            true_label = gallery_labels_orig[query_idx]
            sims = embeddings_orig @ query_emb; sims[query_idx]=-1.0
            top_idxs = np.argsort(sims)[::-1][:ENS_N_TOP]
            top5v = [float(sims[top_idxs[i]]) for i in range(min(5,len(top_idxs)))]
            std_top5_v = float(np.std(top5v)) if len(top5v)>=2 else 0.05
            ts3  = [float(sims[top_idxs[i]]) for i in range(min(3,len(top_idxs)))]
            margin   = ts3[0]-(ts3[1] if len(ts3)>1 else 0)
            gap23    = (ts3[1] if len(ts3)>1 else 0)-(ts3[2] if len(ts3)>2 else 0)
            cen_sims_all = cen_matrix @ query_emb
            cen_order    = np.argsort(cen_sims_all)[::-1]
            cen_rank_map = {cen_labels[i]:r for r,i in enumerate(cen_order)}
            cen_sorted   = np.sort(cen_sims_all)[::-1]
            q_cen_gap    = compute_centroid_gap(query_emb, cen_sorted)
            q_mar_ratio  = compute_margin_ratio(margin, true_label, median_margins)
            for j in top_idxs:
                cand_lbl = gallery_labels_orig[int(j)]
                if cand_lbl==true_label: continue
                total_neg+=1
                knn_sim = float(sims[j])
                cen_sim = float(np.dot(query_emb, centroids.get(cand_lbl,query_emb)))
                n_img   = float(label_counts.get(cand_lbl,1))
                shg = float(bool(
                    set(label_to_hard_groups.get(cand_lbl,[])) &
                    set(label_to_hard_groups.get(true_label,[]))
                ))
                cen_rk = cen_rank_map.get(cand_lbl,len(cen_labels))
                co_members = group_coMembers.get(cand_lbl,set())
                if co_members:
                    co_sims = [(m,float(np.dot(query_emb,centroids.get(m,query_emb))))
                               for m in co_members if m in centroids]
                    co_sims.sort(key=lambda x:-x[1])
                    intra_rk = next((r for r,(m,_) in enumerate(co_sims) if m==cand_lbl),len(co_sims))
                else: intra_rk=0
                q_rank_cons = compute_rank_consistency(cand_lbl, 0, cen_rank_map)
                feat = extract_features_v36(
                    knn_sim, cen_sim, margin, 0, n_img, 0.5, 0.5, knn_sim,
                    0.0, 0.0, shg, 1.0, 0.1, gap23, 0.0, cen_rk, intra_rk, 0, std_top5_v,
                    q_cen_gap, q_mar_ratio, q_rank_cons)
                X_s  = scaler.transform(np.array([feat],dtype=np.float32))
                p_rf = float(rf_model.predict_proba(X_s)[0][col_pos_rf_c])
                p_et = float(et_model.predict_proba(X_s)[0][col_pos_et_c])
                prob = float(stack_model.predict_proba(np.array([[p_rf,p_et]]))[0][1])
                if prob>=ENS_MIN_PROB: false_pos+=1
        if total_neg>0: fp_rate[lbl]=false_pos/total_neg
    unknown_boost = {}
    for lbl,fpr in fp_rate.items():
        boost = UNKNOWN_BOOST_DEFAULT + fpr*(UNKNOWN_BOOST_MAX-UNKNOWN_BOOST_DEFAULT)*2
        unknown_boost[lbl] = float(np.clip(boost, UNKNOWN_BOOST_DEFAULT, UNKNOWN_BOOST_MAX))
    n_b = sum(1 for b in unknown_boost.values() if b>UNKNOWN_BOOST_DEFAULT+0.01)
    log(f"  {n_b}/{len(unknown_boost)} labels with elevated boost")
    return unknown_boost


def build_ensemble_v36(embeddings_orig, gallery_labels_orig, label_to_idxs_orig,
                        centroids, visual_sigs, label_counts, hard_groups, label_to_hard_groups,
                        median_margins, cen_labels, cen_matrix):
    log("Building ensemble v36 (22 features)...")
    X, y, w = [], [], []
    rng = np.random.default_rng(42)
    group_members = {gn:[m for m in members if m in centroids]
                     for gn,members in hard_groups.items()}
    for lbl, idxs in tqdm(label_to_idxs_orig.items(), desc="ENS v36"):
        if len(idxs)<2: continue
        my_hg = label_to_hard_groups.get(lbl,[])
        for query_idx in idxs:
            query_emb  = embeddings_orig[query_idx]
            true_label = gallery_labels_orig[query_idx]
            sims = embeddings_orig @ query_emb
            sims[query_idx] = -1.0
            top_idxs = np.argsort(sims)[::-1][:ENS_N_TOP+5]
            cand_best = {}
            for j in top_idxs:
                if j==query_idx: continue
                cl=gallery_labels_orig[j]; sf=float(sims[j])
                if cl not in cand_best or sf>cand_best[cl]: cand_best[cl]=sf
            ranked = sorted(cand_best.items(), key=lambda x:-x[1])
            if not ranked: continue
            top5v      = [s for _,s in ranked[:5]]
            std_top5_v = float(np.std(top5v)) if len(top5v)>=2 else 0.05
            top1_sim   = ranked[0][1]
            top2_sim   = ranked[1][1] if len(ranked)>1 else 0.0
            top3_sim   = ranked[2][1] if len(ranked)>2 else 0.0
            margin_v   = top1_sim - top2_sim
            gap23      = top2_sim - top3_sim
            cen_sims_all = cen_matrix @ query_emb
            cen_order    = np.argsort(cen_sims_all)[::-1]
            cen_rank_map = {cen_labels[i]:r for r,i in enumerate(cen_order)}
            cen_sorted   = np.sort(cen_sims_all)[::-1]
            q_cen_gap = compute_centroid_gap(query_emb, cen_sorted)
            q_margin_ratio = compute_margin_ratio(margin_v, true_label, median_margins)
            for rank_i,(cand_lbl,knn_sim) in enumerate(ranked[:ENS_N_TOP]):
                cen_sim    = float(np.dot(query_emb, centroids.get(cand_lbl,query_emb)))
                n_img      = float(label_counts.get(cand_lbl,1))
                is_correct = int(cand_lbl==true_label)
                cand_groups= set(label_to_hard_groups.get(cand_lbl,[]))
                shg = float(bool(cand_groups & set(my_hg)) and cand_lbl!=true_label)
                cen_rk  = cen_rank_map.get(cand_lbl, len(cen_labels))
                co_members = group_coMembers.get(cand_lbl, set())
                if co_members:
                    co_sims = [(m,float(np.dot(query_emb,centroids.get(m,query_emb))))
                               for m in co_members if m in centroids]
                    co_sims.sort(key=lambda x:-x[1])
                    intra_rk = next((r for r,(m,_) in enumerate(co_sims) if m==cand_lbl), len(co_sims))
                else: intra_rk = 0
                q_rank_cons = compute_rank_consistency(cand_lbl, rank_i, cen_rank_map)
                r = rng.random()
                if r<0.35 and is_correct: ocr_cf,ocr_mf,ocr_ex = 1.0,1.0,1
                elif r<0.55: ocr_cf=0.5; ocr_mf=float(rng.random()>0.4); ocr_ex=0
                else: ocr_cf=0.0; ocr_mf=0.0; ocr_ex=0
                feat = extract_features_v36(
                    knn_sim, cen_sim, margin_v, rank_i, n_img,
                    0.5, 0.5, top1_sim, ocr_cf, ocr_mf, shg, 1.0, 0.1,
                    gap23, rank_i/max(ENS_N_TOP-1,1), cen_rk, intra_rk, ocr_ex, std_top5_v,
                    q_cen_gap, q_margin_ratio, q_rank_cons)
                X.append(feat); y.append(is_correct); w.append(1.0)
            if my_hg:
                for gname in my_hg:
                    confusers = [m for m in group_members.get(gname,[]) if m!=true_label]
                    for confuser in confusers[:3]:
                        conf_knn = float(np.dot(query_emb, centroids[confuser]))
                        cen_rk   = cen_rank_map.get(confuser, len(cen_labels))
                        q_rank_cons = compute_rank_consistency(confuser, 0, cen_rank_map)
                        feat = extract_features_v36(
                            conf_knn, conf_knn, margin_v, 0,
                            float(label_counts.get(confuser,1)),
                            0.5, 0.5, top1_sim, 0.0, 0.0, 1.0, 1.0, 0.1,
                            gap23, 0.0, cen_rk, 0, 0, std_top5_v,
                            q_cen_gap, q_margin_ratio, q_rank_cons)
                        X.append(feat); y.append(0); w.append(ENS_HARD_NEG_WEIGHT)
    log(f"  {len(X)} examples | {sum(y)} positives | {len(X)-sum(y)} negatives")
    return (np.array(X,dtype=np.float32), np.array(y,dtype=np.int32), np.array(w,dtype=np.float32))

In [9]:
# ══ OCR init + Ensemble load/train + do_query (dino-v36 verbatim) ═══════════

# ── OCR reader ────────────────────────────────────────────────────────────────
OCR_AVAILABLE = False
ocr_reader = None
if OCR_IMPORT_OK:
    try:
        ocr_reader = easyocr.Reader(OCR_LANGS, gpu=(device=='cuda'), verbose=False)
        OCR_AVAILABLE = True
        log("OCR OK")
    except Exception as e:
        log(f"OCR unavailable: {e}")

# ── Ensemble load or train ────────────────────────────────────────────────────
need_regen = True
ood_gap_thr = OOD_GAP_REJECT_THR

if os.path.exists(ENSEMBLE_PKL):
    try:
        with open(ENSEMBLE_PKL,'rb') as f: ens_data = pickle.load(f)
        if ens_data.get('version')=='v36' and ens_data.get('n_features')==N_FEATURES:
            need_regen    = False
            rf_model      = ens_data['rf']
            et_model      = ens_data['et']
            stack_model   = ens_data['stack']
            scaler        = ens_data['scaler']
            unknown_boost = ens_data.get('unknown_boost',{})
            ood_gap_thr   = ens_data.get('ood_gap_thr', OOD_GAP_REJECT_THR)
            col_pos_rf    = list(rf_model.classes_).index(1) if 1 in rf_model.classes_ else 1
            col_pos_et    = list(et_model.classes_).index(1) if 1 in et_model.classes_ else 1
            col_pos_stack = list(stack_model.classes_).index(1) if 1 in stack_model.classes_ else 1
            log("Ensemble v36 loaded from cache")
    except Exception as e:
        log(f"Ensemble cache error: {e}"); need_regen = True

if need_regen:
    ood_gap_thr = calibrate_ood_gap_threshold(
        emb_mat, gallery_labels_orig, label_to_idxs,
        centroids, cen_labels_global, cen_matrix_global,
        gap_scores_known, target_fpr=OOD_TARGET_FPR)

    X_train, y_train, w_train = build_ensemble_v36(
        emb_mat, gallery_labels_orig, label_to_idxs,
        centroids, visual_sigs, label_counts, hard_groups, label_to_hard_groups,
        median_margins, cen_labels_global, cen_matrix_global)

    log(f"Training RF({RF_N_ESTIMATORS}) + ET({ET_N_ESTIMATORS}) + Stack...")
    scaler = StandardScaler(); X_scaled = scaler.fit_transform(X_train)

    rf_model = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF, max_features='sqrt', n_jobs=-1, random_state=42)
    rf_model.fit(X_scaled, y_train, sample_weight=w_train)
    col_pos_rf = list(rf_model.classes_).index(1) if 1 in rf_model.classes_ else 1

    et_model = ExtraTreesClassifier(n_estimators=ET_N_ESTIMATORS,
        min_samples_leaf=ET_MIN_SAMPLES_LEAF, max_features='sqrt', n_jobs=-1, random_state=123)
    et_model.fit(X_scaled, y_train, sample_weight=w_train)
    col_pos_et = list(et_model.classes_).index(1) if 1 in et_model.classes_ else 1

    p_rf = rf_model.predict_proba(X_scaled)[:,col_pos_rf]
    p_et = et_model.predict_proba(X_scaled)[:,col_pos_et]
    stack_model = LogisticRegression(C=STACK_C, max_iter=STACK_MAX_ITER, random_state=0)
    stack_model.fit(np.column_stack([p_rf,p_et]), y_train, sample_weight=w_train)
    col_pos_stack = list(stack_model.classes_).index(1) if 1 in stack_model.classes_ else 1

    unknown_boost = calibrate_unknown_boost(
        rf_model, et_model, stack_model, scaler,
        emb_mat, gallery_labels_orig, label_to_idxs,
        centroids, label_counts, label_to_hard_groups,
        median_margins, cen_labels_global, cen_matrix_global)

    with open(ENSEMBLE_PKL,'wb') as f:
        pickle.dump({'version':'v36','n_features':N_FEATURES,
                     'rf':rf_model,'et':et_model,'stack':stack_model,
                     'scaler':scaler,'unknown_boost':unknown_boost,
                     'ood_gap_thr':ood_gap_thr,'median_margins':median_margins}, f)
    log(f"Ensemble v36 saved → {ENSEMBLE_PKL}")

log("All models ready.")

# ── do_query (dino-v36 verbatim) ──────────────────────────────────────────────
def do_query(crop, box_w=100.0, box_h=100.0, frame_h=1000.0, frame_w=1000.0):
    emb = encode_tta(crop)
    if emb is None: return [],'?',0.0,0.0,0.0,0.0,'none'

    k=min(K_NEIGHBORS,faiss_index.ntotal)
    sims,idx=faiss_index.search(emb[np.newaxis],k=k)
    knn_best={}
    for s,i in zip(sims[0],idx[0]):
        if int(i)==-1: continue
        lbl=gallery_labels[int(i)]; sf=float(s)
        if lbl not in knn_best or sf>knn_best[lbl]: knn_best[lbl]=sf
    if not knn_best: return [],'?',0.0,0.0,0.0,0.0,'none'

    ranked_knn = sorted(knn_best.items(),key=lambda x:-x[1])
    top1_lbl,top1_sim = ranked_knn[0]
    top2_sim   = ranked_knn[1][1] if len(ranked_knn)>1 else 0.0
    top3_sim   = ranked_knn[2][1] if len(ranked_knn)>2 else 0.0
    margin     = top1_sim - top2_sim
    knn_gap_23 = top2_sim - top3_sim
    top5_labels= [lbl for lbl,_ in ranked_knn[:ENS_N_TOP]]
    topk = [(l,round(s,4)) for l,s in ranked_knn[:TOP_K_SHOW]]

    top5v      = [s for _,s in ranked_knn[:5]]
    std_top5_v = float(np.std(top5v)) if len(top5v)>=2 else 0.05
    box_wh_ratio  = float(box_w/(box_h+1e-6))
    box_area_frac = float((box_w*box_h)/max(frame_h*frame_w,1))

    cen_sims_all = cen_matrix_global @ emb
    cen_order    = np.argsort(cen_sims_all)[::-1]
    cen_rank_map = {cen_labels_global[i]:r for r,i in enumerate(cen_order)}
    cen_sorted   = np.sort(cen_sims_all)[::-1]
    cen_gap = compute_centroid_gap(emb, cen_sorted)
    mar_ratio = compute_margin_ratio(margin, top1_lbl, median_margins)

    # [FIX-1] UNKNOWN strict 2 signals
    if top1_sim < UNKNOWN_STRICT2_KNN and margin < UNKNOWN_STRICT2_MARGIN:
        log(f"    [UNK-S2] knn={top1_sim:.3f} m={margin:.4f} -> REJECT")
        return topk,'?',round(margin,4),round(top1_sim,4),round(margin,4),round(UNKNOWN_STRICT2_KNN,4),'unk_strict2'

    # [OOD v34]
    if (ood_gap_thr > OOD_MIN_FLOOR
            and cen_gap < ood_gap_thr
            and margin < OOD_MARGIN_REJECT
            and top1_sim < OOD_KNN_REJECT):
        return topk,'?',round(cen_gap,4),round(top1_sim,4),round(margin,4),round(ood_gap_thr,4),'ood_gap_reject'

    ocr_label,ocr_conf,ocr_method,ts_score,text_norm = ocr_match_full(crop, top5_labels)
    ocr_exact = 1 if ts_score>=OCR_EXACT_TOKEN_THR else 0

    ocr_top3_boost = None
    if text_norm and len(text_norm)>=3:
        for lbl,_ in ranked_knn[:3]:
            lbl_norm = GALLERY_NAMES_NORM.get(lbl,'')
            ts = fuzz.token_sort_ratio(text_norm, lbl_norm)
            if ts >= OCR_BOOST_THR:
                ocr_top3_boost = lbl; break

    feat_list,lbl_list = [],[]
    hsv_scores = {}
    for rank_i,(cand_lbl,knn_sim) in enumerate(ranked_knn[:ENS_N_TOP]):
        cen_sim = float(np.dot(emb,centroids.get(cand_lbl,emb)))
        n_img   = float(label_counts.get(cand_lbl,1))
        hsv_s,lbp_s = get_visual_scores(crop,cand_lbl)
        hsv_scores[cand_lbl] = 0.6*hsv_s+0.4*lbp_s
        ocr_match_flag = 1.0 if ocr_label==cand_lbl else 0.0
        query_groups   = set(label_to_hard_groups.get(top1_lbl,[]))
        cand_groups    = set(label_to_hard_groups.get(cand_lbl,[]))
        shg = float(bool(query_groups & cand_groups) and cand_lbl!=top1_lbl)
        cen_rk     = cen_rank_map.get(cand_lbl,len(cen_labels_global))
        co_members = group_coMembers.get(cand_lbl,set())
        if co_members:
            co_sims = [(m,float(np.dot(emb,centroids.get(m,emb)))) for m in co_members if m in centroids]
            co_sims.sort(key=lambda x:-x[1])
            intra_rk = next((r for r,(m,_) in enumerate(co_sims) if m==cand_lbl),len(co_sims))
        else: intra_rk=0
        q_rank_cons = compute_rank_consistency(cand_lbl, rank_i, cen_rank_map)
        feat=extract_features_v36(
            knn_sim,cen_sim,margin,rank_i,n_img,hsv_s,lbp_s,top1_sim,
            ocr_conf,ocr_match_flag,shg,box_wh_ratio,box_area_frac,
            knn_gap_23,rank_i/max(ENS_N_TOP-1,1),cen_rk,intra_rk,ocr_exact,std_top5_v,
            cen_gap, mar_ratio, q_rank_cons)
        feat_list.append(feat); lbl_list.append(cand_lbl)

    if not feat_list:
        return topk,'?',0.0,round(top1_sim,4),round(margin,4),global_thr,'none'

    X_pred   = scaler.transform(np.array(feat_list,dtype=np.float32))
    p_rf_all = rf_model.predict_proba(X_pred)[:,col_pos_rf]
    p_et_all = et_model.predict_proba(X_pred)[:,col_pos_et]
    probs    = stack_model.predict_proba(np.column_stack([p_rf_all,p_et_all]))[:,col_pos_stack]

    best_idx = np.argmax(probs)
    pred_lbl = lbl_list[best_idx]
    ens_prob = float(probs[best_idx])

    if ocr_conf>=0.5 and ocr_label==pred_lbl:
        mode = f'ens+ocr({ocr_method})'
    elif ocr_conf>=0.5 and ocr_label!=pred_lbl:
        mode = f'ens+ocr_dis({ocr_method})'
        if ocr_conf>=OCR_VETO_MIN_CONF and ens_prob<OCR_VETO_MAX_PROB and ocr_label:
            pred_lbl=ocr_label; ens_prob=ocr_conf*0.88
            mode=f'ocr_veto({ocr_method})'
    else:
        mode='ens_vis'

    if ocr_top3_boost and ocr_top3_boost != pred_lbl and mode not in ('ocr_veto(token_sort)','ocr_veto(partial)'):
        pred_lbl = ocr_top3_boost; mode = 'ocr_top3_boost'

    dino_score = top1_sim * (1.0 + MARGIN_WEIGHT * margin)

    if len(ranked_knn)>=2:
        top2_lbl    = ranked_knn[1][0]
        top1_groups = set(label_to_hard_groups.get(pred_lbl,[]))
        top2_groups = set(label_to_hard_groups.get(top2_lbl,[]))
        if (top1_groups & top2_groups) and crop is not None and margin<0.05:
            hr_scores = {cl: (hsv_similarity_hr(crop,cl)
                              if cl in (group_coMembers.get(pred_lbl,set())|{pred_lbl})
                              else hsv_scores.get(cl,0.5))
                         for cl in lbl_list}
            hr_best_lbl,hr_best_score = max(hr_scores.items(),key=lambda x:x[1])
            hr_current = hr_scores.get(pred_lbl,0.5)
            if hr_best_lbl!=pred_lbl and hr_best_score>hr_current+0.08:
                mode+='+hr_rerank'; pred_lbl=hr_best_lbl

    hsv_top1      = hsv_scores.get(pred_lbl,0.5)
    hsv_max_other = max((v for l,v in hsv_scores.items() if l!=pred_lbl),default=0.5)
    if hsv_top1>hsv_max_other+0.08: hsv_adj=HSV_BONUS_MAX*(hsv_top1-hsv_max_other)
    elif hsv_top1<hsv_max_other-0.12: hsv_adj=-HSV_PENALTY
    else: hsv_adj=0.0

    if ens_prob>=ENS_MIN_PROB:
        final_score=dino_score*(1.0+hsv_adj)
        final_score=max(final_score,ens_prob*0.90)
    else:
        pred_lbl=top1_lbl; final_score=dino_score*(1.0+hsv_adj*0.5)
        mode='dino_fallback'

    thr = per_label_thr.get(pred_lbl,global_thr)
    thr = min(thr,THR_ABSOLUTE_MAX)
    margin_factor = np.clip(margin/MARGIN_THR_REF, 0.0, 1.0)
    thr_eff = thr * (1.0 - MARGIN_THR_SCALE * margin_factor)
    thr_eff = max(thr_eff, 0.50)

    if margin <= UNKNOWN_STRICT_MARGIN and top1_sim < UNKNOWN_STRICT_KNN:
        thr_eff = min(thr_eff * UNKNOWN_STRICT_BOOST, THR_ABSOLUTE_MAX)
        mode += '+unk_strict'

    if final_score >= thr_eff:
        if top1_sim < thr_eff:
            boost      = unknown_boost.get(pred_lbl,UNKNOWN_BOOST_DEFAULT)
            strict_thr = thr_eff * boost
            if final_score < strict_thr:
                return topk,'?',round(final_score,4),round(top1_sim,4),round(margin,4),round(strict_thr,4),mode
        return topk,pred_lbl,round(final_score,4),round(top1_sim,4),round(margin,4),round(thr_eff,4),mode

    if margin>=MARGIN_MIN_FALLBACK and top1_sim>=FALLBACK_BASE_KNN:
        thr_fb  = min(per_label_thr.get(top1_lbl,global_thr),THR_ABSOLUTE_MAX)
        thr_fbm = thr_fb*(1.0-0.5*min(margin,0.10))
        if top1_sim>=thr_fbm:
            return topk,top1_lbl,round(top1_sim,4),round(top1_sim,4),round(margin,4),round(thr_fbm,4),'fallback'

    return topk,'?',round(final_score,4),round(top1_sim,4),round(margin,4),round(thr_eff,4),mode

[11:14:59] OCR OK
[11:14:59] Calibrating centroid_gap threshold (target FPR=0.03)...
[11:15:00]   gap_known  : mean=0.0665 p5=0.0081
[11:15:00]   gap_unknown: mean=0.0334 p95=0.0798
[11:15:00]   OOD gap threshold = 0.0100
[11:15:00] Building ensemble v36 (22 features)...


ENS v36: 100%|██████████| 313/313 [00:04<00:00, 73.88it/s] 


[11:15:04]   3784 examples | 1921 positives | 1863 negatives
[11:15:04] Training RF(400) + ET(400) + Stack...
[11:15:06] Calibrating unknown boost...
[11:16:17]   39/120 labels with elevated boost
[11:16:17] Ensemble v36 saved → /kaggle/working/ensemble_v36.pkl
[11:16:17] All models ready.


In [10]:
# ══ OBBUtils + parse_detections (yolo-deepsort-cnn-v1-version-stable verbatim) ═

class OBBUtils:
    @staticmethod
    def obb_to_hbb(cx, cy, w, h, angle_rad, img_w, img_h):
        cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
        corners = np.array([[-w/2,-h/2],[w/2,-h/2],[w/2,h/2],[-w/2,h/2]])
        R = np.array([[cos_a, -sin_a],[sin_a, cos_a]])
        corners = corners @ R.T + np.array([cx, cy])
        return (int(corners[:,0].min()), int(corners[:,1].min()),
                int(corners[:,0].max()), int(corners[:,1].max()))

    @staticmethod
    def draw_obb(img, cx, cy, w, h, angle_rad, color, thickness=2):
        cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
        corners = np.array([[-w/2,-h/2],[w/2,-h/2],[w/2,h/2],[-w/2,h/2]])
        R = np.array([[cos_a, -sin_a],[sin_a, cos_a]])
        corners = (corners @ R.T + np.array([cx, cy])).astype(np.int32)
        for i in range(4):
            cv2.line(img, tuple(corners[i]), tuple(corners[(i+1)%4]), color, thickness)

def parse_detections(yolo_results):
    dets = []
    for r in yolo_results:
        if r.obb is None or len(r.obb) == 0:
            continue
        for i, (cx, cy, w, h, angle) in enumerate(r.obb.xywhr.cpu().numpy()):
            dets.append({
                "cx":         float(cx),
                "cy":         float(cy),
                "w":          float(w),
                "h":          float(h),
                "angle":      float(angle),
                "conf":       float(r.obb.conf[i]),
                "class_id":   int(r.obb.cls[i]),
                "class_name": CLASS_NAMES.get(int(r.obb.cls[i]), "product"),
            })
    return dets

In [11]:
# ══ TrackCropBuffer + OBBTracker (yolo-deepsort-cnn-v1-version-stable verbatim) ═

class TrackCropBuffer:
    def __init__(self, max_crops=5):
        self.max_crops = max_crops
        self.buffers   = defaultdict(list)

    def add(self, stable_id, conf_ema, crop):
        if crop is None:
            return
        self.buffers[stable_id].append((conf_ema, crop))
        if len(self.buffers[stable_id]) > self.max_crops:
            self.buffers[stable_id].sort(reverse=True, key=lambda x: x[0])
            self.buffers[stable_id] = self.buffers[stable_id][:self.max_crops]

    def best_crop(self, stable_id):
        buf = self.buffers.get(stable_id)
        return max(buf, key=lambda x: x[0])[1] if buf else None

    def clear(self, stable_id=None):
        if stable_id is None:
            self.buffers.clear()
        else:
            self.buffers.pop(stable_id, None)


class OBBTracker:
    def __init__(self):
        self.tracker = DeepSort(
            max_age=30,
            n_init=3,
            max_iou_distance=0.7,
            max_cosine_distance=0.4,
            nn_budget=500,
            embedder="mobilenet",
            embedder_gpu=False,
        )
        self.id_map         = {}
        self.next_stable_id = 1
        self.banned_stable_ids = set()
        self.lost_tracks       = {}
        self.stable_position_history = defaultdict(list)
        self.position_history        = defaultdict(list)
        self.raw_id_creation_frame   = {}
        self.exited_raw_ids          = {}
        self.crop_buffer      = TrackCropBuffer(max_crops=5)
        self.conf_ema         = defaultdict(lambda: 0.0)
        self.confirmed_tracks = set()
        self.ALPHA                = 0.33
        self.IOU_THRESHOLD        = 0.40
        self.VELOCITY_HISTORY_LEN = 15
        self.MIN_EXIT_FRAMES      = 60
        self.frame_count          = 0

    def _ban_stable_id(self, sid, raw_id):
        if sid is None or sid in self.banned_stable_ids:
            return
        self.lost_tracks.pop(sid, None)
        self.stable_position_history.pop(sid, None)
        self.crop_buffer.clear(sid)
        self.conf_ema.pop(sid, None)
        self.banned_stable_ids.add(sid)
        if raw_id is not None:
            self.exited_raw_ids[raw_id] = self.frame_count
            self.position_history.pop(raw_id, None)
            self.raw_id_creation_frame.pop(raw_id, None)
            self.id_map.pop(raw_id, None)
        log(f"PRODUIT SORTI -> stable_id {sid} BANNI (raw {raw_id})")

    def _recycle_raw_id(self, raw_id):
        self.position_history.pop(raw_id, None)
        self.raw_id_creation_frame[raw_id] = self.frame_count
        log(f"raw_id {raw_id} recycle (frame {self.frame_count})")

    def update_conf_ema(self, stable_id, conf):
        self.conf_ema[stable_id] = (
            self.ALPHA * conf + (1 - self.ALPHA) * self.conf_ema[stable_id]
        )
        return self.conf_ema[stable_id]

    @staticmethod
    def extract_obb_crop(frame, cx, cy, w, h, angle_rad, padding=0.22):
        ih, iw = frame.shape[:2]
        M       = cv2.getRotationMatrix2D((cx, cy), np.degrees(angle_rad), 1.0)
        rotated = cv2.warpAffine(frame, M, (iw, ih), flags=cv2.INTER_LINEAR)
        pw  = int(w * (1 + padding))
        ph  = int(h * (1 + padding))
        x1  = max(0, int(cx - pw / 2))
        y1  = max(0, int(cy - ph / 2))
        x2  = min(iw, x1 + pw)
        y2  = min(ih, y1 + ph)
        crop = rotated[y1:y2, x1:x2]
        if crop.size == 0:
            return None
        return cv2.resize(crop, (224, 224), interpolation=cv2.INTER_AREA)

    def _compute_cosine_similarity(self, crop1, crop2):
        if crop1 is None or crop2 is None:
            return 0.0
        try:
            emb1 = self.tracker.embedder([crop1])[0]
            emb2 = self.tracker.embedder([crop2])[0]
            return float(
                np.dot(emb1, emb2) /
                (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-8)
            )
        except Exception:
            return 0.0

    def _get_or_reuse_stable_id(self, raw_id, cx, cy, angle, w, h, iw, ih, current_crop):
        if raw_id in self.id_map:
            sid = self.id_map[raw_id]
            if sid not in self.banned_stable_ids:
                return sid
            else:
                log(f"raw_id {raw_id} pointed to banned sid {sid} -> recalculate")
                del self.id_map[raw_id]
                self._recycle_raw_id(raw_id)

        diag        = np.hypot(iw, ih)
        reid_thresh = diag * 0.30
        best_sid         = None
        best_score       = -1.0
        best_frames_lost = 0

        for sid, info in list(self.lost_tracks.items()):
            if sid in self.banned_stable_ids:
                self.lost_tracks.pop(sid, None)
                continue
            dist = np.hypot(cx - info['cx'], cy - info['cy'])
            if dist >= reid_thresh * 1.2:
                continue
            frames_lost   = self.frame_count - info['frame']
            spatial_score = max(0, 1.0 - (dist / reid_thresh))
            if frames_lost < 30 and spatial_score > 0.65:
                log(f"ReID fast-path (occlusion {frames_lost}f) -> stable_id {sid}")
                self.id_map[raw_id] = sid
                self.lost_tracks.pop(sid, None)
                return sid
            angle_diff = (
                min(abs(angle - info['angle']),
                    180 - abs(angle - info['angle'])) / 180.0
            )
            size_diff        = abs(w * h - info['area']) / (w * h + 1e-6)
            last_crop        = self.crop_buffer.best_crop(sid)
            appearance_score = self._compute_cosine_similarity(current_crop, last_crop)
            appear_weight  = max(0.25, 0.55 - (frames_lost / 2000.0) * 0.30)
            spatial_weight = min(0.60, 0.30 + (frames_lost / 2000.0) * 0.30)
            angle_weight   = 0.10
            size_weight    = 0.05
            total_w = appear_weight + spatial_weight + angle_weight + size_weight
            appear_weight  /= total_w
            spatial_weight /= total_w
            angle_weight   /= total_w
            size_weight    /= total_w
            score = (spatial_score      * spatial_weight
                     + appearance_score * appear_weight
                     + (1 - angle_diff) * angle_weight
                     + (1 - size_diff)  * size_weight)
            if score > best_score:
                best_score       = score
                best_sid         = sid
                best_frames_lost = frames_lost

        if best_sid is not None:
            adaptive_thresh = max(0.45, REID_MIN_SCORE - min(0.35, best_frames_lost / 1500.0))
            if best_score >= adaptive_thresh:
                self.id_map[raw_id] = best_sid
                self.lost_tracks.pop(best_sid, None)
                log(f"ReID OK -> stable_id {best_sid} (score {best_score:.3f})")
                return best_sid

        stable_id            = self.next_stable_id
        self.id_map[raw_id]  = stable_id
        self.next_stable_id += 1
        log(f"New stable_id {stable_id} (raw {raw_id})")
        return stable_id

    def _add_stable_position(self, stable_id, cx, cy):
        hist = self.stable_position_history[stable_id]
        if hist:
            last = hist[-1]
            frames_elapsed = max(1, self.frame_count - last['frame'])
            dist  = np.hypot(cx - last['cx'], cy - last['cy'])
            speed = dist / frames_elapsed
            if speed > MAX_REALISTIC_SPEED:
                return False
        hist.append({'cx': cx, 'cy': cy, 'frame': self.frame_count})
        if len(hist) > 30:
            hist.pop(0)
        return True

    def update(self, frame, detections):
        self.frame_count += 1
        ih, iw = frame.shape[:2]
        hbb_dets = []
        for d in detections:
            if d['conf'] < CONF_THRESHOLD:
                continue
            x1, y1, x2, y2 = OBBUtils.obb_to_hbb(
                d['cx'], d['cy'], d['w'], d['h'], d['angle'], iw, ih
            )
            hbb_dets.append(([x1, y1, x2 - x1, y2 - y1], d['conf'], 'product'))

        tracks = self.tracker.update_tracks(hbb_dets, frame=frame)
        results    = []
        active_raw = set()

        for t in tracks:
            if not t.is_confirmed():
                continue
            raw_id = t.track_id
            if raw_id in self.exited_raw_ids:
                frames_since_exit = self.frame_count - self.exited_raw_ids[raw_id]
                if frames_since_exit < EXITED_RAW_TTL:
                    continue
                else:
                    log(f"raw_id {raw_id} released after TTL ({frames_since_exit} frames)")
                    del self.exited_raw_ids[raw_id]
                    self._recycle_raw_id(raw_id)
            if raw_id in self.id_map:
                old_sid = self.id_map[raw_id]
                if old_sid in self.banned_stable_ids:
                    del self.id_map[raw_id]
                    self._recycle_raw_id(raw_id)

            tb       = t.to_ltrb()
            best_det = None
            max_iou  = 0.0
            for d in detections:
                if d['conf'] < CONF_THRESHOLD:
                    continue
                db  = OBBUtils.obb_to_hbb(
                    d['cx'], d['cy'], d['w'], d['h'], d['angle'], iw, ih
                )
                ix1  = max(tb[0], db[0])
                iy1  = max(tb[1], db[1])
                ix2  = min(tb[2], db[2])
                iy2  = min(tb[3], db[3])
                inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
                union = (
                    (tb[2] - tb[0]) * (tb[3] - tb[1])
                    + (db[2] - db[0]) * (db[3] - db[1])
                    - inter
                )
                iou = inter / union if union > 0 else 0
                if iou > max_iou:
                    max_iou  = iou
                    best_det = d

            if best_det and max_iou > self.IOU_THRESHOLD:
                cx    = best_det['cx']
                cy    = best_det['cy']
                angle = best_det['angle']
                w     = best_det['w']
                h     = best_det['h']
                crop  = self.extract_obb_crop(frame, cx, cy, w, h, angle)
                stable_id = self._get_or_reuse_stable_id(
                    raw_id, cx, cy, angle, w, h, iw, ih, crop
                )
                conf_ema = self.update_conf_ema(stable_id, best_det['conf'])
                self.crop_buffer.add(stable_id, conf_ema, crop)
                results.append({
                    'track_id': stable_id,
                    'det':      best_det,
                    'bbox':     tb,
                    'crop':     crop,
                    'conf_ema': round(conf_ema, 3),
                })
                if raw_id not in self.raw_id_creation_frame:
                    self.raw_id_creation_frame[raw_id] = self.frame_count
                self.position_history[raw_id].append({
                    'cx': cx, 'cy': cy, 'angle': angle,
                    'w': w, 'h': h, 'area': w * h,
                    'frame': self.frame_count,
                })
                if len(self.position_history[raw_id]) > self.VELOCITY_HISTORY_LEN:
                    self.position_history[raw_id].pop(0)
                added = self._add_stable_position(stable_id, cx, cy)
                if not added:
                    log(f"   raw {raw_id}/sid {stable_id}: position filtered (double OBB or artefact)")
                active_raw.add(raw_id)

        for tid in list(self.position_history.keys()):
            if tid in active_raw:
                continue
            if tid in self.exited_raw_ids:
                continue
            last_frame      = self.position_history[tid][-1]['frame']
            frames_inactive = self.frame_count - last_frame
            stable          = self.id_map.get(tid)
            if stable and stable not in self.banned_stable_ids:
                last = self.position_history[tid][-1]
                if stable not in self.lost_tracks:
                    frames_seen = self.frame_count - self.raw_id_creation_frame.get(
                        tid, self.frame_count
                    )
                    if frames_seen >= MIN_FRAMES_BEFORE_LOST:
                        self.lost_tracks[stable] = {
                            'cx':    last['cx'],
                            'cy':    last['cy'],
                            'angle': last['angle'],
                            'w':     last['w'],
                            'h':     last['h'],
                            'area':  last['area'],
                            'frame': last['frame'],
                        }
                        log(f"lost_tracks: stable_id {stable} stored (frame {self.frame_count})")
                if frames_inactive > self.MIN_EXIT_FRAMES:
                    if self._should_ban_as_exit(tid, stable, iw, ih, frames_inactive):
                        self._ban_stable_id(stable, tid)
            elif frames_inactive > self.MIN_EXIT_FRAMES:
                self.position_history.pop(tid, None)
                self.id_map.pop(tid, None)
            if frames_inactive > 2400 and tid not in self.exited_raw_ids:
                self.position_history.pop(tid, None)
                self.raw_id_creation_frame.pop(tid, None)

        for sid in list(self.lost_tracks.keys()):
            age = self.frame_count - self.lost_tracks[sid]['frame']
            if age > MAX_LOST_AGE:
                self.lost_tracks.pop(sid, None)
                self.stable_position_history.pop(sid, None)
                log(f"lost_track sid {sid} expired ({age} frames)")

        for rid in list(self.exited_raw_ids.keys()):
            if self.frame_count - self.exited_raw_ids[rid] > EXITED_RAW_TTL:
                del self.exited_raw_ids[rid]
                self.position_history.pop(rid, None)
                self.raw_id_creation_frame.pop(rid, None)

        for s in [s for s in list(self.lost_tracks.keys()) if s in self.banned_stable_ids]:
            self.lost_tracks.pop(s, None)
            self.stable_position_history.pop(s, None)
            log(f"Guard: banned sid {s} removed from lost_tracks")

        return results

    def _should_ban_as_exit(self, raw_id, stable_id, iw, ih, frames_inactive):
        positions = self.position_history.get(raw_id, [])
        lost_info = self.lost_tracks.get(stable_id)
        if not positions and not lost_info:
            return frames_inactive > 800
        if positions and len(positions) >= 1:
            recent_n = min(5, len(positions))
            recent   = positions[-recent_n:]
            mean_cx  = np.mean([p['cx'] for p in recent])
            mean_cy  = np.mean([p['cy'] for p in recent])
        else:
            mean_cx = lost_info['cx']
            mean_cy = lost_info['cy']
        stable_hist = self.stable_position_history.get(stable_id, [])
        was_moving  = False
        if len(stable_hist) >= 6:
            dx = stable_hist[-1]['cx'] - stable_hist[-6]['cx']
            dy = stable_hist[-1]['cy'] - stable_hist[-6]['cy']
            was_moving = np.hypot(dx, dy) > 8.0
        if not was_moving and frames_inactive < 120:
            return False
        margin     = EXIT_MARGIN_DYNAMIC if was_moving else EXIT_MARGIN_STATIC
        near_border = (
            mean_cx < iw * margin
            or mean_cx > iw * (1 - margin)
            or mean_cy < ih * margin
            or mean_cy > ih * (1 - margin)
        )
        if near_border:
            return True
        if frames_inactive > 300:
            return True
        if len(stable_hist) >= 8:
            recent_pos = stable_hist[-4:]
            older_pos  = stable_hist[-8:-4]
            mean_cx_r  = np.mean([p['cx'] for p in recent_pos])
            mean_cx_o  = np.mean([p['cx'] for p in older_pos])
            mean_cy_r  = np.mean([p['cy'] for p in recent_pos])
            mean_cy_o  = np.mean([p['cy'] for p in older_pos])
            vx = mean_cx_r - mean_cx_o
            vy = mean_cy_r - mean_cy_o
            speed = np.hypot(vx, vy)
            if speed > 2.0 and speed <= MAX_REALISTIC_SPEED:
                projected_cx = mean_cx_r + vx * 60
                projected_cy = mean_cy_r + vy * 60
                heading_to_border = (
                    projected_cx < iw * EXIT_MARGIN_DYNAMIC * 2
                    or projected_cx > iw * (1 - EXIT_MARGIN_DYNAMIC * 2)
                    or projected_cy < ih * EXIT_MARGIN_DYNAMIC * 2
                    or projected_cy > ih * (1 - EXIT_MARGIN_DYNAMIC * 2)
                )
                if heading_to_border:
                    return True
        return False

    def get_new_confirmations(self, tracks):
        new = []
        for t in tracks:
            sid = t.get('track_id')
            if sid and sid not in self.confirmed_tracks:
                self.confirmed_tracks.add(sid)
                new.append(sid)
        return new

In [12]:
# ══ RecognitionAdapter (File 1 structure, _query bridges to do_query)
# ══ SessionCounter (NEW — cross-frame product counting) ═════════════════════

class RecognitionAdapter:
    def __init__(self, threshold=0.50):
        self._thresh      = threshold
        self._state       = {}
        self._frame       = 0
        self._prev_active = set()   # sids active in previous enrich() call

    def enrich(self, tracks, banned_ids=None):
        self._frame += 1
        if banned_ids:
            for sid in list(self._state.keys()):
                if sid in banned_ids:
                    del self._state[sid]

        active_sids = set(t['track_id'] for t in tracks)

        # ── CORE FIX: unlock state the moment a track leaves the active set ──
        # Any stable_id that was active last frame but is gone now just had its
        # product removed from the basket. Reset recognition so whatever enters
        # this slot next (same product returning OR a different product via ReID)
        # always gets fresh DINO queries — never inherits a locked identity.
        for sid in (self._prev_active - active_sids):
            st = self._state.get(sid)
            if st and st['locked']:
                log(f"Track #{sid} left frame — recognition unlocked for next entry")
                st['locked']       = False
                st['product_name'] = '?'
                st['best_sim']     = 0.0
                st['votes']        = defaultdict(int)
                st['n_queries']    = 0
                st['last_query_f'] = -RECOGNITION_REQUERY_INTERVAL

        self._prev_active = active_sids

        for t in tracks:
            sid  = t['track_id']
            crop = t.get('crop')

            st = self._state.setdefault(sid, {
                'locked':       False,
                'product_name': '?',
                'best_sim':     0.0,
                'votes':        defaultdict(int),
                'n_queries':    0,
                'last_query_f': -RECOGNITION_REQUERY_INTERVAL,
            })

            if not st['locked']:
                frames_since_query = self._frame - st['last_query_f']
                should_query = (
                    crop is not None
                    and frames_since_query >= RECOGNITION_REQUERY_INTERVAL
                    and st['n_queries'] < RECOGNITION_MAX_VOTES
                )

                if should_query:
                    product, sim = self._query(crop)
                    st['last_query_f'] = self._frame
                    st['n_queries']   += 1

                    if sim >= self._thresh:
                        st['votes'][product] += 1

                    if sim > st['best_sim']:
                        st['best_sim']     = sim
                        st['product_name'] = product if sim >= self._thresh else '?'

                    if st['votes']:
                        mv = max(st['votes'], key=st['votes'].get)
                        st['product_name'] = mv

                    if sim >= RECOGNITION_CONF_LOCK:
                        st['locked'] = True
                        log(f"Track #{sid} locked -> '{st['product_name']}' (sim={sim:.3f})")

                    elif st['n_queries'] >= RECOGNITION_MAX_VOTES and st['votes']:
                        st['locked'] = True
                        log(f"Track #{sid} locked by majority -> '{st['product_name']}'")

            t['product_name']   = st['product_name']
            t['product_sim']    = round(st['best_sim'], 3)
            t['product_locked'] = st['locked']

        return tracks

    def _query(self, bgr_crop):
        if bgr_crop is None:
            return '?', 0.0
        try:
            pil_crop = Image.fromarray(cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB))
            _, pred_lbl, final_score, _, _, _, _ = do_query(pil_crop)
            return pred_lbl, float(final_score)
        except Exception as e:
            log(f"_query error: {e}")
            return '?', 0.0

class SessionCounter:
    '''
    Tracks product counts across the entire shopping session.
    - When a track exits with a confirmed label -> finalized[label] += 1
    - Active (in-frame) locked tracks count as +1 each while present
    - get_receipt() returns the merged count for the tablet
    '''

    def __init__(self):
        self.finalized = {}   # {label: count}  — exited confirmed products
        self.active    = {}   # {stable_id: label} — currently in frame

    def on_lock(self, stable_id, label):
        if label and label != '?':
            self.active[stable_id] = label

    def on_exit(self, stable_id):
        label = self.active.pop(stable_id, None)
        if label and label != '?':
            self.finalized[label] = self.finalized.get(label, 0) + 1

    def get_receipt(self):
        receipt = dict(self.finalized)
        for label in self.active.values():
            if label and label != '?':
                receipt[label] = receipt.get(label, 0) + 1
        return receipt

    def reset(self):
        self.finalized.clear()
        self.active.clear()

In [13]:
# ══ YOLO model loading + draw_frame (yolo-deepsort-cnn-v1-version-stable verbatim) ═

log(f"Loading YOLO: {YOLO_PT}")
model = YOLO(YOLO_PT)
CLASS_NAMES = model.names
log(f"  {len(CLASS_NAMES)} classes")


def draw_frame(frame, detections, tracks):
    ih, iw = frame.shape[:2]

    for d in detections:
        cx = d["cx"] * iw if d["cx"] <= 1 else d["cx"]
        cy = d["cy"] * ih if d["cy"] <= 1 else d["cy"]
        w  = d["w"]  * iw if d["w"]  <= 1 else d["w"]
        h  = d["h"]  * ih if d["h"]  <= 1 else d["h"]
        OBBUtils.draw_obb(frame, cx, cy, w, h, d["angle"], (0, 200, 255), 1)

    for t in tracks:
        d   = t["det"]
        tid = int(t["track_id"])
        cx  = d["cx"] * iw if d["cx"] <= 1 else d["cx"]
        cy  = d["cy"] * ih if d["cy"] <= 1 else d["cy"]
        w   = d["w"]  * iw if d["w"]  <= 1 else d["w"]
        h   = d["h"]  * ih if d["h"]  <= 1 else d["h"]
        color = (int(tid * 50) % 255, int(tid * 100) % 255, int(tid * 150) % 255)
        OBBUtils.draw_obb(frame, cx, cy, w, h, d.get("angle", 0), color, 4)

        prod_name = t.get('product_name', '?')
        prod_sim  = t.get('product_sim',  0.0)
        locked    = t.get('product_locked', False)

        lock_icon = ' L' if locked else ''
        top_label = f"#{tid} {prod_name}{lock_icon}"
        bot_label = f"sim={prod_sim:.2f}" if prod_name != '?' else 'identifying...'

        cv2.putText(frame, top_label,
                    (int(cx) - 40, int(cy) - 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.62, color, 2)
        cv2.putText(frame, bot_label,
                    (int(cx) - 40, int(cy) - 14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 1)

    return frame

[11:16:17] Loading YOLO: /kaggle/input/datasets/sarahlaouedj25/yoloresultobbv5/best(2).pt
[11:16:18]   1 classes


In [14]:
# ══ HTML template (yolo-deepsort-cnn-v1-version-stable verbatim) ════════════

HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
<title>SmartBasket AI Server</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Rajdhani:wght@600&display=swap');
  :root {
    --green: #00ff88; --dim: #003322; --bg: #060a0f;
    --panel: #0d1117; --border: #1a2e1a; --red: #ff3355;
    --amber: #ffaa00; --blue: #00ccff;
  }
  * { box-sizing: border-box; margin:0; padding:0; }
  body {
    background: var(--bg); color: var(--green);
    font-family: 'Share Tech Mono', monospace;
    display: grid; grid-template-rows: auto 1fr auto;
    min-height: 100vh; padding: 12px; gap: 12px; font-size: 16px;
  }
  header { display: flex; align-items: center; justify-content: space-between;
    flex-wrap: wrap; gap: 10px; border-bottom: 1px solid var(--border); padding-bottom: 12px; }
  header h1 { font-family:'Rajdhani',sans-serif; font-size: 1.45rem; letter-spacing: 2px; }
  .version-badge { font-size: 0.75rem; color: var(--blue); margin-left: 8px;
    border: 1px solid var(--blue); padding: 2px 6px; border-radius: 4px; }
  #ws-status { display:flex; align-items:center; gap:8px; font-size:0.85rem;
    padding: 8px 16px; border-radius:20px; border:1px solid currentColor; }
  #ws-status .dot { width:9px; height:9px; border-radius:50%; background:currentColor;
    animation: pulse 1.5s infinite; }
  @keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }
  #ws-status.connected { color: var(--green); }
  #ws-status.disconnected { color: var(--red); animation: none; }
  main { display: grid; grid-template-columns: 1fr; gap: 12px; min-height: 0; }
  #canvas-wrap { position: relative; width: 100%; aspect-ratio: 4 / 3;
    border: 2px solid var(--border); border-radius: 12px; overflow: hidden; background: #000;
    display: flex; align-items: center; justify-content: center; }
  #canvas-wrap canvas { width: 100% !important; height: 100% !important; object-fit: contain; }
  #overlay-msg { position:absolute; inset:0; display:flex; align-items:center; justify-content:center;
    font-size:1.1rem; color:#445; pointer-events:none; transition: opacity .4s; }
  aside { display: flex; flex-direction: column; gap: 12px; }
  .panel { background: var(--panel); border: 1px solid var(--border); border-radius: 10px; padding: 14px; }
  .panel h2 { font-family: 'Rajdhani', sans-serif; font-size: 1.1rem;
    letter-spacing: 2px; margin-bottom: 12px; color: #88ffcc; }
  .stats { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }
  .stat-box { background: var(--dim); border-radius: 8px; padding: 12px 8px; text-align: center; }
  .stat-box .val { font-size: 1.45rem; font-weight: 700; line-height: 1; }
  .stat-box .lbl { font-size: 0.7rem; color: #446655; margin-top: 4px; letter-spacing: 1px; }
  button { padding: 14px 16px; min-height: 52px; border: 1px solid var(--green);
    background: transparent; color: var(--green); font-family: 'Share Tech Mono', monospace;
    font-size: 1rem; border-radius: 8px; cursor: pointer; transition: all .2s; width: 100%; margin-bottom: 8px; }
  button:hover:not(:disabled) { background: var(--green); color: #000; }
  button:disabled { opacity: 0.35; cursor: not-allowed; }
  button.danger { border-color: var(--red); color: var(--red); }
  button.danger:hover:not(:disabled) { background: var(--red); color: white; }
  #feed { display: flex; flex-direction: column; gap: 8px;
    max-height: 160px; overflow-y: auto; padding-right: 4px; }
  .feed-item { background: var(--dim); border-left: 3px solid var(--green);
    padding: 8px 12px; border-radius: 0 6px 6px 0; font-size: 0.85rem; animation: slideIn 0.3s ease; }
  .feed-item.reid   { border-left-color: var(--amber); }
  .feed-item.recog  { border-left-color: var(--blue);  }
  @keyframes slideIn { from { opacity:0; transform: translateX(-10px); } to { opacity:1; transform: none; } }
  #product-table { font-size: 0.78rem; width: 100%; border-collapse: collapse; }
  #product-table th { text-align: left; color: #446655; padding: 2px 4px; border-bottom: 1px solid var(--border); }
  #product-table td { padding: 4px 4px; vertical-align: middle; }
  .locked-dot { display:inline-block; width:7px; height:7px;
    border-radius:50%; background:var(--blue); margin-right:4px; }
  #receipt-table { font-size: 0.78rem; width: 100%; border-collapse: collapse; }
  #receipt-table th { text-align: left; color: #446655; padding: 2px 4px; border-bottom: 1px solid var(--border); }
  #receipt-table td { padding: 4px 4px; vertical-align: middle; }
  footer { border-top: 1px solid var(--border); padding-top: 10px;
    font-size: 0.78rem; color: #334;
    display: flex; justify-content: space-between; flex-wrap: wrap; gap: 8px; }
  @media (min-width: 768px) {
    main { grid-template-columns: 1fr 310px; }
    #feed { max-height: 200px; }
    header h1 { font-size: 1.7rem; }
  }
</style>
</head>
<body>
<header>
  <h1>SMARTBASKET <span class="version-badge">DINO-V36</span></h1>
  <div id="ws-status" class="disconnected">
    <span class="dot"></span><span id="ws-label">DISCONNECTED</span>
  </div>
</header>
<main>
  <div id="canvas-wrap">
    <canvas id="canvas"></canvas>
    <div id="overlay-msg">&#9654; Press START to begin streaming</div>
  </div>
  <aside>
    <div class="panel">
      <h2>CONTROLS</h2>
      <button id="btnStart" onclick="startCamera()">&#9654; START CAMERA</button>
      <button id="btnStop" onclick="stopCamera()" disabled class="danger">&#9632; STOP</button>
      <button onclick="flipCamera()">&#x21C4; FLIP CAMERA</button>
      <button onclick="resetSession()" style="border-color:var(--amber);color:var(--amber)">&#x21BA; RESET SESSION</button>
    </div>
    <div class="panel">
      <h2>METRICS</h2>
      <div class="stats">
        <div class="stat-box"><div class="val" id="stat-fps">&#8212;</div><div class="lbl">FPS</div></div>
        <div class="stat-box"><div class="val" id="stat-tracks">&#8212;</div><div class="lbl">TRACKS</div></div>
        <div class="stat-box"><div class="val" id="stat-dets">&#8212;</div><div class="lbl">DETS</div></div>
        <div class="stat-box"><div class="val" id="stat-inf">&#8212;</div><div class="lbl">INF ms</div></div>
        <div class="stat-box" style="grid-column:span 2">
          <div class="val" id="stat-rtt">&#8212;</div><div class="lbl">RTT ms</div>
        </div>
      </div>
    </div>
    <div class="panel">
      <h2>LIVE TRACKS</h2>
      <table id="product-table">
        <thead><tr><th>Track</th><th>Product</th><th>Sim</th></tr></thead>
        <tbody id="product-body"></tbody>
      </table>
    </div>
    <div class="panel">
      <h2>SESSION RECEIPT</h2>
      <table id="receipt-table">
        <thead><tr><th>Product</th><th>Qty</th></tr></thead>
        <tbody id="receipt-body"></tbody>
      </table>
    </div>
    <div class="panel" style="flex:1">
      <h2>EVENTS</h2>
      <div id="feed"></div>
    </div>
  </aside>
</main>
<footer>
  <span>OBB DeepSORT + DINOv2 Recognition &bull; SmartBasket</span>
  <span id="frame-counter">frames: 0</span>
</footer>
<script>
let ws = null, stream = null, running = false, animId = null;
let facingMode = 'environment';
let framesSent = 0, sendTimestamp = 0;
const video = document.createElement('video');
video.playsInline = true; video.muted = true;
const canvas = document.getElementById('canvas');
const ctx = canvas.getContext('2d');

function connectWS() {
  const proto = location.protocol === 'https:' ? 'wss' : 'ws';
  ws = new WebSocket(`${proto}://${location.host}/ws`);
  ws.onopen = () => setStatus(true);
  ws.onmessage = (evt) => {
    const rtt = Date.now() - sendTimestamp;
    const data = JSON.parse(evt.data);
    if (data.image) {
      const img = new Image();
      img.onload = () => {
        canvas.width = img.naturalWidth; canvas.height = img.naturalHeight;
        ctx.drawImage(img, 0, 0);
        document.getElementById('overlay-msg').style.opacity = '0';
      };
      img.src = 'data:image/jpeg;base64,' + data.image;
    }
    document.getElementById('stat-fps').textContent    = data.fps ?? '—';
    document.getElementById('stat-tracks').textContent = data.tracks ?? '—';
    document.getElementById('stat-dets').textContent   = data.dets ?? '—';
    document.getElementById('stat-inf').textContent    = data.inference_ms ?? '—';
    document.getElementById('stat-rtt').textContent    = rtt;
    (data.new_confirmations || []).forEach(id => addFeedItem(id, 'confirm'));
    updateProductTable(data.products || {});
    updateReceiptTable(data.session_receipt || {});
  };
  ws.onclose = () => { setStatus(false); if (running) setTimeout(connectWS, 2000); };
  ws.onerror = () => ws.close();
}

function updateProductTable(products) {
  const tbody = document.getElementById('product-body');
  const existing = {};
  for (const row of tbody.rows) existing[row.dataset.tid] = row;
  const seen = new Set();
  for (const [tid, info] of Object.entries(products)) {
    seen.add(String(tid));
    let row = existing[tid];
    if (!row) {
      row = tbody.insertRow(); row.dataset.tid = tid;
      row.insertCell(); row.insertCell(); row.insertCell();
      if (info.locked && info.name !== '?') addFeedItem(tid, 'recog', info.name, info.sim);
    }
    const lockDot = info.locked ? '<span class="locked-dot"></span>' : '';
    row.cells[0].textContent = '#' + tid;
    row.cells[1].innerHTML   = lockDot + (info.name || '?');
    row.cells[2].textContent = info.sim ? info.sim.toFixed(2) : '—';
  }
  for (const [tid, row] of Object.entries(existing)) {
    if (!seen.has(tid)) tbody.removeChild(row);
  }
}

function updateReceiptTable(receipt) {
  const tbody = document.getElementById('receipt-body');
  tbody.innerHTML = '';
  for (const [label, count] of Object.entries(receipt)) {
    const row = tbody.insertRow();
    row.insertCell().textContent = label;
    row.insertCell().textContent = count;
  }
}

function setStatus(connected) {
  const el = document.getElementById('ws-status');
  const lbl = document.getElementById('ws-label');
  el.className    = connected ? 'connected' : 'disconnected';
  lbl.textContent = connected ? 'CONNECTED' : 'DISCONNECTED';
}

async function openStream() {
  if (stream) stream.getTracks().forEach(t => t.stop());
  stream = await navigator.mediaDevices.getUserMedia({
    video: { facingMode: { ideal: facingMode }, width: { ideal: 640 }, height: { ideal: 480 } }
  });
  video.srcObject = stream;
  await video.play();
}

async function startCamera() {
  await openStream(); running = true; connectWS();
  document.getElementById('btnStart').disabled = true;
  document.getElementById('btnStop').disabled  = false;
  captureLoop();
}

function stopCamera() {
  running = false;
  if (animId) cancelAnimationFrame(animId);
  if (stream) stream.getTracks().forEach(t => t.stop());
  if (ws) ws.close();
  ctx.clearRect(0, 0, canvas.width, canvas.height);
  document.getElementById('overlay-msg').style.opacity = '1';
  document.getElementById('btnStart').disabled = false;
  document.getElementById('btnStop').disabled  = true;
}

async function flipCamera() {
  facingMode = facingMode === 'user' ? 'environment' : 'user';
  if (stream) await openStream();
}

async function resetSession() {
  await fetch('/reset', {method:'POST'});
  document.getElementById('receipt-body').innerHTML = '';
  addFeedItem(0, 'confirm', 'Session reset', 0);
}

function captureLoop() {
  if (!running) return;
  if (video.readyState >= 2 && ws && ws.readyState === WebSocket.OPEN) {
    const tmp = document.createElement('canvas');
    tmp.width  = video.videoWidth  || 640;
    tmp.height = video.videoHeight || 480;
    tmp.getContext('2d').drawImage(video, 0, 0);
    const quality = /Mobi|Android|iPhone/i.test(navigator.userAgent) ? 0.68 : 0.78;
    const b64 = tmp.toDataURL('image/jpeg', quality).split(',')[1];
    sendTimestamp = Date.now();
    ws.send(JSON.stringify({ image: b64 }));
    framesSent++;
    document.getElementById('frame-counter').textContent = 'frames: ' + framesSent;
  }
  animId = requestAnimationFrame(captureLoop);
}

function addFeedItem(id, type, extra, sim) {
  const feed = document.getElementById('feed');
  const el   = document.createElement('div');
  let text;
  if (type === 'recog')
    text = 'Track #' + id + ' -> ' + extra + ' (' + (sim||0).toFixed(2) + ') ' + new Date().toLocaleTimeString();
  else
    text = 'Track #' + id + ' confirmed ' + new Date().toLocaleTimeString();
  el.className = 'feed-item ' + type;
  el.textContent = text;
  feed.prepend(el);
  while (feed.children.length > 20) feed.lastChild.remove();
}
</script>
</body>
</html>
"""

In [15]:
# ══ Hot-reload gallery (bridge) ═══════════════════════════════════════════════
_gallery_reload_lock = threading.Lock()

def reload_gallery_data():
    global gallery_labels, gallery_labels_orig, DIM, use_multiscale
    global centroids, visual_sigs, global_thr_raw, per_label_thr_raw
    global label_counts, label_to_idxs, gallery_prods, faiss_index
    global per_label_thr, global_thr, cen_labels_global, cen_matrix_global
    global cen_matrix, cen_labels, hard_groups, label_to_hard_groups
    global group_coMembers, GALLERY_NAMES_NORM, emb_mat, median_margins
    global gap_scores_known, gap_arr

    with _gallery_reload_lock:
        log("[Reload] Starting gallery hot-reload...")

        # Step 1: wait for Kaggle to finish processing the new version, then download
        time.sleep(10)
        try:
            _kaggle_mod.api.authenticate()
            _kaggle_mod.api.dataset_download_files(
                _GALLERY_DATASET,
                path=_GALLERY_DOWNLOAD_DIR,
                unzip=True, force=True, quiet=True
            )
            log("[Reload] ✅ Latest files downloaded")
        except Exception as e:
            log(f"[Reload] ⚠️ Download failed ({e}) — reloading from existing files")

        # Step 2: reload PKL + FAISS
        pkl_path   = _gallery_path("gallery_finetuned_v3.pkl")
        faiss_path = _gallery_path("gallery_finetuned_v3.index")
        try:
            with open(pkl_path, "rb") as f:
                gdata = pickle.load(f)

            gallery_labels      = gdata.get("all_labels", gdata["labels"])
            gallery_labels_orig = gdata.get("labels", gallery_labels)
            DIM                 = gdata["embed_dim"]
            use_multiscale      = gdata.get("use_multiscale", False)
            centroids           = gdata.get("centroids", {})
            visual_sigs         = gdata.get("visual_signatures", {})
            global_thr_raw      = gdata.get("global_thr", 0.734)
            per_label_thr_raw   = gdata.get("per_label_thr", {})
            label_counts        = gdata.get("label_counts", {})
            label_to_idxs       = gdata.get("label_to_idxs", {})
            gallery_prods       = gdata.get("products", sorted(set(gallery_labels)))

            faiss_index = faiss.read_index(faiss_path)
            try: faiss_index.make_direct_map()
            except: pass

            # Step 3: recompute derived structures
            per_label_thr = {}
            for lbl, thr in per_label_thr_raw.items():
                cnt = label_counts.get(lbl, 1)
                t = thr * SINGLETON_THR_FACTOR if cnt == 1 else thr
                per_label_thr[lbl] = round(min(t, THR_HARD_CAP), 4)
            global_thr = round(min(global_thr_raw * GLOBAL_THR_FACTOR, THR_HARD_CAP), 4)

            cen_labels_global = sorted(centroids.keys())
            cen_matrix_global = (np.stack([centroids[l] for l in cen_labels_global]).astype(np.float32)
                                 if cen_labels_global else np.zeros((0, DIM), dtype=np.float32))
            cen_matrix = cen_matrix_global
            cen_labels = cen_labels_global

            hard_groups, label_to_hard_groups = compute_hard_groups(centroids)
            group_coMembers = defaultdict(set)
            for gname, members in hard_groups.items():
                for m in members:
                    for other in members:
                        if other != m: group_coMembers[m].add(other)

            GALLERY_NAMES_NORM = {lbl: re.sub(r"[^a-z0-9]", "", lbl.lower()) for lbl in gallery_prods}

            _n = faiss_index.ntotal
            emb_mat = np.zeros((_n, DIM), dtype=np.float32)
            for i in range(_n):
                try: emb_mat[i] = faiss_index.reconstruct(i)
                except: pass

            median_margins = {}
            for lbl, idxs in label_to_idxs.items():
                if len(idxs) < 2: median_margins[lbl] = 0.05; continue
                margins = []
                for idx in idxs:
                    emb = emb_mat[idx].astype(np.float32)
                    sims_all = cen_matrix @ emb
                    sorted_sims = np.sort(sims_all)[::-1]
                    margins.append(float(sorted_sims[0] - sorted_sims[1]) if len(sorted_sims) > 1 else 0.05)
                median_margins[lbl] = float(np.median(margins))

            gap_scores_known = []
            for lbl, idxs in label_to_idxs.items():
                if lbl not in centroids: continue
                for idx in idxs:
                    emb = emb_mat[idx].astype(np.float32)
                    sims_all = cen_matrix @ emb
                    sorted_sims = np.sort(sims_all)[::-1]
                    gap_scores_known.append(compute_centroid_gap(emb, sorted_sims))
            gap_arr = np.array(gap_scores_known)

            # Step 4: delete stale ensemble so it retrains on next query
            if os.path.exists(ENSEMBLE_PKL):
                os.remove(ENSEMBLE_PKL)
                log("[Reload] Stale ensemble deleted — will retrain on next query")

            # Step 5: clear recognizer vote history (stale track/product refs)
            recognizer._state = {}

            log(f"[Reload] ✅ Done — {len(gallery_prods)} products, {faiss_index.ntotal} vectors")
            return {"success": True, "n_products": len(gallery_prods), "n_vectors": faiss_index.ntotal}

        except Exception as e:
            log(f"[Reload] ❌ Failed: {e}")
            return {"success": False, "error": str(e)}

In [16]:
# ══ Global state + Flask app setup ══════════════════════════════════════════

import socket as _socket_mod

def find_free_port(start=5000):
    for p in range(start, start + 100):
        try:
            with _socket_mod.socket(_socket_mod.AF_INET, _socket_mod.SOCK_STREAM) as s:
                s.bind(("0.0.0.0", p)); return p
        except OSError:
            pass
    raise RuntimeError("No free port found")

PORT = find_free_port(5000)
log(f"Port: {PORT}")

app  = Flask(__name__)
sock = Sock(app)
CORS(app)

# ── Queues, locks, FPS state ─────────────────────────────────────────────────
frame_queue = queue.Queue(maxsize=1)
ws_clients  = set()
ws_lock     = threading.Lock()
fps_val     = 0.0
frame_count = 0
last_time   = time.time()

# ── Tracker + recognizer + session counter ───────────────────────────────────
tracker         = OBBTracker()
recognizer      = RecognitionAdapter(threshold=0.50)
session_counter = SessionCounter()
prev_banned     = set()

@app.route("/")
def index():
    return Response(HTML, mimetype="text/html")

@app.route("/reset", methods=["POST"])
def reset_session():
    session_counter.reset()
    return jsonify({"status": "ok"})

@app.route("/status", methods=["GET"])
def get_status():
    return jsonify({"session_receipt": session_counter.get_receipt()})

@app.route("/api/reload-gallery", methods=["POST"])
def api_reload_gallery():
    if request.headers.get("X-Reload-Secret", "") != RELOAD_SECRET:
        return jsonify({"error": "unauthorized"}), 401
    threading.Thread(target=reload_gallery_data, daemon=True).start()
    return jsonify({"status": "reload started"}), 202

@sock.route("/ws")
def ws_handler(ws):
    log(f"Client connected ({id(ws)})")
    with ws_lock:
        ws_clients.add(ws)
    try:
        while True:
            msg = ws.receive()
            if msg is None:
                break
            data  = json.loads(msg)
            frame = cv2.imdecode(
                np.frombuffer(base64.b64decode(data.get("image", "")), np.uint8),
                cv2.IMREAD_COLOR,
            )
            if frame is None:
                continue
            try:
                frame_queue.put_nowait(frame)
            except queue.Full:
                pass
    except Exception as e:
        log(f"WS error: {e}")
    finally:
        with ws_lock:
            ws_clients.discard(ws)
        log(f"Client disconnected ({id(ws)})")

[11:16:18] Port: 5000


In [17]:
# ══ Inference loop (yolo-deepsort-cnn-v1-version-stable + session_receipt) ═══

def push_to_clients(payload):
    dead = set()
    with ws_lock:
        clients = set(ws_clients)
    for ws in clients:
        try:
            ws.send(payload)
        except Exception:
            dead.add(ws)
    if dead:
        with ws_lock:
            ws_clients.difference_update(dead)


def inference_loop():
    global fps_val, frame_count, last_time, prev_banned
    min_interval = 1.0 / TARGET_FPS
    log("Inference thread started")

    while True:
        frame      = frame_queue.get()
        t_start    = time.time()
        frame_count += 1

        now = time.time()
        elapsed = now - last_time
        if elapsed >= 1.0:
            fps_val     = frame_count / elapsed
            frame_count = 0
            last_time   = now

        # Step 1: YOLO detection
        yolo_results = model(frame, conf=CONF_THRESHOLD, verbose=False)
        detections   = parse_detections(yolo_results)

        # Step 2: OBB tracking (unchanged)
        tracks = tracker.update(frame, detections)

        # Step 3: SessionCounter — register newly banned stable_ids
        newly_banned = tracker.banned_stable_ids - prev_banned
        for sid in newly_banned:
            session_counter.on_exit(sid)
        prev_banned = set(tracker.banned_stable_ids)

        # Step 4: Recognition adapter (DINOv2 via do_query)
        tracks = recognizer.enrich(tracks, banned_ids=tracker.banned_stable_ids)

        # Step 5: SessionCounter — register locked labels
        for t in tracks:
            if t.get('product_locked') and t.get('product_name', '?') != '?':
                session_counter.on_lock(t['track_id'], t['product_name'])

        # Step 6: Visualise
        annotated = draw_frame(frame.copy(), detections, tracks)

        _, buf  = cv2.imencode(".jpg", annotated, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        img_b64 = base64.b64encode(buf).decode()

        product_map = {
            t['track_id']: {
                'name':   t.get('product_name', '?'),
                'sim':    t.get('product_sim',  0.0),
                'locked': t.get('product_locked', False),
            }
            for t in tracks
        }

        payload = json.dumps({
            "image":             img_b64,
            "fps":               round(fps_val, 1),
            "tracks":            len(tracks),
            "dets":              len(detections),
            "new_confirmations": tracker.get_new_confirmations(tracks),
            "inference_ms":      round((time.time() - t_start) * 1000, 1),
            "products":          product_map,
            "session_receipt":   session_counter.get_receipt(),
        })
        push_to_clients(payload)

        elapsed = time.time() - t_start
        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)


threading.Thread(target=inference_loop, daemon=True).start()

[11:16:18] Inference thread started


In [18]:
import gc, psutil

def periodic_cleanup():
    """Frees accumulated memory every 60 seconds."""
    while True:
        time.sleep(30)
        try:
            # ── 1. CUDA cache (main cause of hallucination / slowdown) ──────
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

            # ── 2. Python garbage ────────────────────────────────────────────
            gc.collect()

            # ── 3. Recognition cache — drop state for all banned tracks ──────
            banned = set(tracker.banned_stable_ids)
            stale = [sid for sid in list(recognizer._state.keys()) if sid in banned]
            for sid in stale:
                recognizer._state.pop(sid, None)

            # ── 4. Crop buffers — drop stored crops for banned tracks ─────────
            for sid in list(tracker.crop_buffer.buffers.keys()):
                if sid in banned:
                    tracker.crop_buffer.clear(sid)

            # ── 5. Trim banned_stable_ids — keep last 50 only ─────────────────
            # Safe because next_stable_id only goes up, no collision possible
            if len(tracker.banned_stable_ids) > 50:
                tracker.banned_stable_ids = set(
                    sorted(tracker.banned_stable_ids)[-50:]
                )

            # ── 6. Log ────────────────────────────────────────────────────────
            ram  = psutil.Process().memory_info().rss / 1024**2
            rec  = len(recognizer._state)
            ban  = len(tracker.banned_stable_ids)
            lost = len(tracker.lost_tracks)
            msg  = f"[CLEANUP] RAM={ram:.0f}MB | recog_cache={rec} | banned={ban} | lost={lost}"
            if torch.cuda.is_available():
                gpu = torch.cuda.memory_allocated() / 1024**2
                msg += f" | GPU={gpu:.0f}MB"
            log(msg)

        except Exception as e:
            log(f"[CLEANUP] error: {e}")

threading.Thread(target=periodic_cleanup, daemon=True).start()
log("Cleanup thread started — runs every 60s")


[11:16:18] Cleanup thread started — runs every 60s


In [ ]:
# ══ ngrok + run ══════════════════════════════════════════════════════════════

def run_flask():
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(2)

from pyngrok import ngrok
import time

# Force kill any existing ngrok processes
ngrok.kill()
time.sleep(2)  # Give it time to fully shut down
try:
    ngrok.kill()
    time.sleep(1)
    # Replace with your ngrok auth token
    ngrok.set_auth_token("3EZlJc4PerveDn6bn7We84IOlSQ_6tLAskEARHL4Bx25e5wnq")
    tunnel = ngrok.connect(PORT)
    url    = tunnel.public_url
except Exception as e:
    url = f"http://localhost:{PORT}"
    log(f"ngrok: {e}")

print("\n" + "="*90)
print("SmartBasket AI Server — YOLO-OBB + DeepSORT + OBBTracker + DINOv2 Recognition")
print()
print("  PIPELINE:")
print("    Camera -> YOLO-OBB -> OBBTracker (stable IDs, ReID, exit detection)")
print("    -> RecognitionAdapter -> do_query (DINOv2 + Ensemble v36)")
print("    -> SessionCounter -> WebSocket broadcast")
print()
print("  API:")
print("    /ws     WebSocket — send {image: base64} receive {frame, tracks, session_receipt}")
print("    /reset  POST — reset session counter")
print("    /status GET  — current session receipt JSON")
print()
print(f"  WSS URL: {url}/ws")
print(f"  UI:      {url}/")
print("="*90)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    ngrok.kill()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit


                                                                                                    
SmartBasket AI Server — YOLO-OBB + DeepSORT + OBBTracker + DINOv2 Recognition

  PIPELINE:
    Camera -> YOLO-OBB -> OBBTracker (stable IDs, ReID, exit detection)
    -> RecognitionAdapter -> do_query (DINOv2 + Ensemble v36)
    -> SessionCounter -> WebSocket broadcast

  API:
    /ws     WebSocket — send {image: base64} receive {frame, tracks, session_receipt}
    /reset  POST — reset session counter
    /status GET  — current session receipt JSON

  WSS URL: https://underfed-rifling-mummify.ngrok-free.dev/ws
  UI:      https://underfed-rifling-mummify.ngrok-free.dev/
[11:16:48] [CLEANUP] RAM=2034MB | recog_cache=0 | banned=0 | lost=0 | GPU=2613MB
[11:17:19] [CLEANUP] RAM=2034MB | recog_cache=0 | banned=0 | lost=0 | GPU=2613MB
[11:17:49] [CLEANUP] RAM=2034MB | recog_cache=0 | banned=0 | lost=0 | GPU=2613MB
[11:18:19] [CLEANUP] RAM=2034MB | recog_cache=0 | banned=0 | lost=0 | GPU=2613MB

127.0.0.1 - - [12/Jun/2026 11:22:28] "GET /ws HTTP/1.1" 200 -


[11:22:28] WS error: Connection closed: 1005 
[11:22:28] Client disconnected (135548904236736)
[11:22:52] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:23:22] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:23:52] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:24:22] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:24:53] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:25:23] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:25:53] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:26:23] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:26:54] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:27:24] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:27:54] [CLEANUP] RAM=2488MB | recog_cache=0 | banned=0 | lost=0 | GPU=2661MB
[11:28:24] [CL